## 查询板块衍生


In [1]:
import json
from pathlib import Path

import pandas as pd

# =========================
# 1) 读取原始数据
# =========================
pkl_path = Path('cdc灰度验证.pkl')
df_raw = pd.read_pickle(pkl_path)
print('df_raw.shape =', df_raw.shape)
print('df_raw.columns =', list(df_raw.columns))

# 基础字段
base_cols = [
    'apply_id',
    'apply_time',
    'approve_state',
    'credit_limit_amount',
    'use_amount',
    'principal_amount_borrowed',
    'fpd7',
    'spd7',
    'credit_apply_cnt',
    'blind_lend',
]
missing_base = [c for c in base_cols if c not in df_raw.columns]
if missing_base:
    raise ValueError(f'missing base cols in df_raw: {missing_base}')

# 时间字段
# apply_time 形如 '2025-11-25 01:53:44.943'
df_raw['apply_time_dt'] = pd.to_datetime(df_raw['apply_time'], errors='coerce')
df_raw['apply_date'] = df_raw['apply_time_dt'].dt.date

# =========================
# 避免穿越：假定必有 create_time 列，只使用 create_time <= apply_time 的最新一条快照
# =========================
df_raw['create_time_dt'] = pd.to_datetime(df_raw['apply_time'], errors='coerce')

# 仅在两端时间都非空时做严格过滤；否则保留（避免误删）
m_valid = df_raw['create_time_dt'].notna() & df_raw['apply_time_dt'].notna()
before_n = len(df_raw)
df_raw = df_raw.loc[(~m_valid) | (df_raw['create_time_dt'] <= df_raw['apply_time_dt'])].copy()

# 若同一 apply_id 仍有多条，取 create_time_dt 最大（即申请前最近一条）
dup_n = int(df_raw.duplicated('apply_id').sum()) if 'apply_id' in df_raw.columns else 0
if dup_n > 0:
    df_raw = (
        df_raw.sort_values(['apply_id', 'create_time_dt'])
             .drop_duplicates('apply_id', keep='last')
             .reset_index(drop=True)
    )

print(f"[create_time对齐] 假定使用列=create_time; 过滤后行数 {before_n} -> {len(df_raw)}; 去重前重复apply_id条数={dup_n}")

# =========================
# 2) 仔细阅读 response_body：解析 & 结构探查（top-level keys / consultas keys）
# =========================
def parse_response_body(x):
    """将 response_body 解析为 dict。"""
    if x is None:
        return {}
    if isinstance(x, dict):
        return x
    if not isinstance(x, str):
        x = str(x)
    x = x.strip()
    if not x:
        return {}
    obj = json.loads(x)
    # 有时可能是双层 json 字符串
    if isinstance(obj, str):
        obj = json.loads(obj)
    return obj if isinstance(obj, dict) else {}

# 抽样查看顶层 keys & consultas keys
sample_n = min(300, len(df_raw))
top_keys = set()
consultas_keys = set()
for x in df_raw['response_body'].head(sample_n):
    obj = parse_response_body(x)
    top_keys.update(obj.keys())
    for c in obj.get('consultas', []) or []:
        if isinstance(c, dict):
            consultas_keys.update(c.keys())

print('\nresponse_body.top_level_keys(sample):', sorted(top_keys))
print('consultas.keys(sample):', sorted(consultas_keys))

# =========================
# 3) 平铺 consultas：一行一条查询记录
# =========================
records = []

for row in df_raw.itertuples(index=False):
    row_dict = row._asdict()
    apply_id = row_dict.get('apply_id')
    apply_time_dt = row_dict.get('apply_time_dt')
    apply_date = row_dict.get('apply_date')

    obj = parse_response_body(row_dict.get('response_body'))
    consultas = obj.get('consultas', []) or []

    for c in consultas:
        if not isinstance(c, dict):
            continue

        rec = {
            # 主键/时间
            'apply_id': apply_id,
            'apply_time': row_dict.get('apply_time'),
            'apply_time_dt': apply_time_dt,
            'apply_date': apply_date,
            # 标签/业务字段
            'approve_state': row_dict.get('approve_state'),
            'fpd7': row_dict.get('fpd7'),
            'spd7': row_dict.get('spd7'),
            'credit_apply_cnt': row_dict.get('credit_apply_cnt'),
            'blind_lend': row_dict.get('blind_lend'),
            'credit_limit_amount': row_dict.get('credit_limit_amount'),
            'use_amount': row_dict.get('use_amount'),
            'principal_amount_borrowed': row_dict.get('principal_amount_borrowed'),
            # consultas 字段
            'fechaConsulta': c.get('fechaConsulta'),
            'nombreOtorgante': c.get('nombreOtorgante'),
            'tipoCredito': c.get('tipoCredito'),
            'importeCredito': c.get('importeCredito'),
            'claveUnidadMonetaria': c.get('claveUnidadMonetaria'),
            'tipoResponsabilidad': c.get('tipoResponsabilidad'),
        }
        records.append(rec)

consultas_flat = pd.DataFrame.from_records(records)

# 类型清洗（保持温和，不强行报错）
consultas_flat['fechaConsulta_dt'] = pd.to_datetime(consultas_flat['fechaConsulta'], errors='coerce')
consultas_flat['importeCredito_num'] = pd.to_numeric(consultas_flat['importeCredito'], errors='coerce')

print('\nconsultas_flat.shape =', consultas_flat.shape)
print('consultas_flat.apply_id.nunique =', consultas_flat['apply_id'].nunique())
print('consultas_flat.tipoCredito.nunique =', consultas_flat['tipoCredito'].nunique())

# 预览
consultas_flat.head(10)

df_raw.shape = (14963, 13)
df_raw.columns = ['Unnamed: 0', 'apply_id', 'cust_no', 'apply_time', 'approve_state', 'blind_lend', 'credit_apply_cnt', 'credit_limit_amount', 'fpd7', 'principal_amount_borrowed', 'response_body', 'spd7', 'use_amount']
[create_time对齐] 假定使用列=create_time; 过滤后行数 14963 -> 14963; 去重前重复apply_id条数=0

response_body.top_level_keys(sample): ['claveOtorgante', 'consultas', 'creditos', 'declaracionesConsumidor', 'domicilios', 'empleos', 'folioConsulta', 'folioConsultaOtorgante', 'mensajes', 'persona']
consultas.keys(sample): ['claveOtorgante', 'claveUnidadMonetaria', 'fechaConsulta', 'importeCredito', 'nombreOtorgante', 'tipoCredito', 'tipoResponsabilidad']

consultas_flat.shape = (463057, 20)
consultas_flat.apply_id.nunique = 14963
consultas_flat.tipoCredito.nunique = 31


,apply_id,apply_time,apply_time_dt,apply_date,approve_state,fpd7,spd7,credit_apply_cnt,blind_lend,credit_limit_amount,use_amount,principal_amount_borrowed,fechaConsulta,nombreOtorgante,tipoCredito,importeCredito,claveUnidadMonetaria,tipoResponsabilidad,fechaConsulta_dt,importeCredito_num
0,1355047166686003201,2026-03-02 10:44:36.872,2026-03-02 10:44:36.872,2026-03-02,CYCLE_PASS,-1.0,-1.0,1.0,NaN,1200.0,NaN,0.0,2026-03-02,CAMELINAS MECANICAS,M,1000,MX,NaN,2026-03-02,1000
1,1355047166686003201,2026-03-02 10:44:36.872,2026-03-02 10:44:36.872,2026-03-02,CYCLE_PASS,-1.0,-1.0,1.0,NaN,1200.0,NaN,0.0,2026-02-27,SOCIEDADES FINANCIERAS POPULARES,M,1000,MX,NaN,2026-02-27,1000
2,1355047166686003201,2026-03-02 10:44:36.872,2026-03-02 10:44:36.872,2026-03-02,CYCLE_PASS,-1.0,-1.0,1.0,NaN,1200.0,NaN,0.0,2026-02-25,SOCIEDAD FINANCIERA DE OBJETO MULTIPLE,M,1000,MX,NaN,2026-02-25,1000
3,1355047166686003201,2026-03-02 10:44:36.872,2026-03-02 10:44:36.872,2026-03-02,CYCLE_PASS,-1.0,-1.0,1.0,NaN,1200.0,NaN,0.0,2026-02-25,MICROFINANCIERA,M,1000,MX,NaN,2026-02-25,1000
4,1355047166686003201,2026-03-02 10:44:36.872,2026-03-02 10:44:36.872,2026-03-02,CYCLE_PASS,-1.0,-1.0,1.0,NaN,1200.0,NaN,0.0,2026-01-21,SIC,F,0,MX,NaN,2026-01-21,0
5,1355047166686003201,2026-03-02 10:44:36.872,2026-03-02 10:44:36.872,2026-03-02,CYCLE_PASS,-1.0,-1.0,1.0,NaN,1200.0,NaN,0.0,2025-12-26,BANCOS,F,0,MX,NaN,2025-12-26,0
6,1355047166686003201,2026-03-02 10:44:36.872,2026-03-02 10:44:36.872,2026-03-02,CYCLE_PASS,-1.0,-1.0,1.0,NaN,1200.0,NaN,0.0,2025-12-26,SOCIEDADES FINANCIERAS POPULARES,M,1000,MX,NaN,2025-12-26,1000
7,1355047166686003201,2026-03-02 10:44:36.872,2026-03-02 10:44:36.872,2026-03-02,CYCLE_PASS,-1.0,-1.0,1.0,NaN,1200.0,NaN,0.0,2025-12-09,COMPANIA DE PRESTAMO PERSONAL,M,1000,MX,NaN,2025-12-09,1000
8,1355047166686003201,2026-03-02 10:44:36.872,2026-03-02 10:44:36.872,2026-03-02,CYCLE_PASS,-1.0,-1.0,1.0,NaN,1200.0,NaN,0.0,2025-10-26,SOCIEDADES FINANCIERAS POPULARES,M,1000,MX,NaN,2025-10-26,1000
9,1355047166686003201,2026-03-02 10:44:36.872,2026-03-02 10:44:36.872,2026-03-02,CYCLE_PASS,-1.0,-1.0,1.0,NaN,1200.0,NaN,0.0,2025-09-26,SIC,F,0,MX,NaN,2025-09-26,0


In [2]:
import numpy as np

# =========================
# 查询板块特征衍生：准备工作（映射写死在代码里）
# - 时间窗口：30/60/90/180/360/720
# - 截止时间字段：优先 request_time，否则 apply_time
# - nombreOtorgante 编码 -> 大类（英文标签，避免特征名含中文）
# - tipoCredito 编码 -> 大类（英文标签，避免特征名含中文）
# - 未命中编码统一归到 Other
# =========================

WINDOW_DAYS = [30, 60, 90, 180, 360, 720, 36000]  # 36000d ~= ALL (覆盖全部历史记录)
DEFAULT_BIG_CLASS = 'Other'

# 截止时间字段：优先 request_time，否则 apply_time
cutoff_col = 'request_time' if 'request_time' in df_raw.columns else 'apply_time'
print('cutoff_col =', cutoff_col)

df_raw['cutoff_dt'] = pd.to_datetime(df_raw[cutoff_col], errors='coerce')

# =========================
# nombreOtorgante 编码 -> 大类（英文标签）
# =========================
nombreOtorgante_to_big = {
    'SIC': 'SIC',
    'COMPANIA DE PRESTAMO PERSONAL': 'PersonalLoanCo',
    'SOFOL PRESTAMO PERSONAL': 'PersonalLoanCo',
    'SOCIEDAD FINANCIERA DE OBJETO MULTIPLE': 'MultiPurposeFin',
    'SOFOM-ENR': 'MultiPurposeFin',
    'SOFOM-ER': 'MultiPurposeFin',
    'MICROFINANCIERA': 'MicroCredit',
    'TELEFONIA CELULAR': 'ServiceCredit',
    'COMUNICACIONES': 'ServiceCredit',
    'VENTA POR CATALOGO': 'ServiceCredit',
    'REPORTE DE CREDITO ESPECIAL': 'ServiceCredit',
    'SERVICIOS': 'ServiceCredit',
    'SERVS. GRALES.': 'ServiceCredit',
    'FINANCIERA': 'FinanceCo',
    'SOCIEDADES FINANCIERAS POPULARES': 'FinanceCo',
    'COOPERATIVA': 'FinanceCo',
    'CAJAS DE AHORRO': 'FinanceCo',
    'COOPERATIVA DE AHORRO Y CREDITO': 'FinanceCo',
    'CIA Q  OTORGA': 'FinanceCo',
    'SEGURO': 'OtherFinance',
    'BANCO': 'Bank',
    'BANCOS': 'Bank',
    'CAMELINAS MECANICAS': 'Halo',
    'TIENDA COMERCIAL': 'Consumer',
    'TIENDA DE AUTOSERVICIO': 'Consumer',
    'CONSUMIDOR FINAL': 'Consumer',
    'TIENDA DEPARTAMENTAL': 'Consumer',
    'TIENDA DE ROPA': 'Consumer',
    'INSUMOS': 'Consumer',
    'MERCANCIA PARA HOGAR Y OFICINA': 'Consumer',
    'MERCANCIA PARA LA CONSTRUCCION': 'Consumer',
    'COMPANIA DE FINANCIAMIENTO DE MOTOCICLET': 'Auto',
    'COMPANIA DE FINANCIAMIENTO AUTOMOTRIZ': 'Auto',
    'AUTOMOTRIZ': 'Auto',
    'OTROS VEHICULOS': 'Auto',
    'FONDOS Y FIDEIC': 'FundTrust',
    'FONDOS Y FIDEICOMISOS': 'FundTrust',
    'BIENES RAICES': 'RealEstate',
    'ARRENDADORAS NO FINANCIERAS': 'NonFinance',
    'ARRENDAMIENTO': 'OtherFinance',
    'AUTOFINANCIAMIENTO': 'OtherFinance',
    'CASA DE EMPENO': 'OtherFinance',
    'FIANZAS': 'OtherFinance',
    'FACTORAJE': 'OtherFinance',
    'HIPOTECARIA': 'OtherFinance',
    'HIPOTECARIO NO BANCARIO': 'OtherFinance',
    'ARRENDADORAS FINANCIERAS': 'OtherFinance',
    'SEGUROS Y FIANZAS': 'OtherFinance',
    'SALUD Y SERVICIOS MEDICOS': 'Medical',
    'SERVICIO MEDICO': 'Medical',
    'GUBERNAMENTALES': 'Government',
    'GOBIERNO': 'Government',
    'EDUCACION': 'Education',
}

# =========================
# tipoCredito 编码 -> 大类（英文标签）
# =========================
tipoCredito_to_big = {
    # Auto
    'CA': 'Auto',
    'AA': 'Auto',
    # Enterprise
    'PM': 'Enterprise',
    'AE': 'Enterprise',
    # OtherLoan
    'PN': 'OtherLoan',
    'NG': 'OtherLoan',
    'PE': 'OtherLoan',
    'A': 'OtherLoan',
    'AR': 'OtherLoan',
    'CF': 'OtherLoan',
    'LR': 'OtherLoan',
    'H': 'OtherLoan',
    'L': 'OtherLoan',
    'AM': 'OtherLoan',
    'P': 'OtherLoan',
    'OT': 'OtherLoan',
    # Credit
    'R': 'Credit',
    'E': 'Credit',
    'PQ': 'Credit',
    'LC': 'Credit',
    # RealEstate
    'BR': 'RealEstate',
    # Bond
    'FI': 'Bond',
    'CO': 'Bond',
    # Guarantee
    'TG': 'Guarantee',
    'PG': 'Guarantee',
    # Others
    'M': 'GeneralPersonalLoan',
    'F': 'FixedPay',
    'TC': 'CreditCard',
    'NC': 'GeneralLoan',
    'PP': 'PersonalLoan',
    'Q': 'UnsecuredCredit',
}

# =========================
# 反向索引：大类 -> 编码列表（后续生成特征时会用到）
# =========================

def invert_mapping(m: dict):
    out = {}
    for code, big in m.items():
        out.setdefault(big, []).append(code)
    for big in out:
        out[big] = sorted(set(out[big]))
    return out

no_big_to_codes = invert_mapping(nombreOtorgante_to_big)
tc_big_to_codes = invert_mapping(tipoCredito_to_big)

# =========================
# 兜底：任何不在映射表里的编码，都统一归到 Other
# （这一步同时对 nombreOtorgante / tipoCredito 生效，不会漏）
# =========================
consultas_flat['nombreOtorgante_code'] = consultas_flat['nombreOtorgante'].astype(str).str.strip()
consultas_flat['tipoCredito_code'] = consultas_flat['tipoCredito'].astype(str).str.strip()

# 生成“已归类后的大类”字段（后续衍生特征建议直接用这两列）
consultas_flat['nombreOtorgante_big'] = consultas_flat['nombreOtorgante_code'].map(nombreOtorgante_to_big).fillna(DEFAULT_BIG_CLASS)
consultas_flat['tipoCredito_big'] = consultas_flat['tipoCredito_code'].map(tipoCredito_to_big).fillna(DEFAULT_BIG_CLASS)

# 未命中编码清单（仅用于检查/排错）
missing_no = sorted(set(consultas_flat['nombreOtorgante_code'].dropna()) - set(nombreOtorgante_to_big.keys()))
missing_tc = sorted(set(consultas_flat['tipoCredito_code'].dropna()) - set(tipoCredito_to_big.keys()))

# 把缺失编码登记到 big_to_codes['Other']，保证后续遍历产出“Other”特征列时覆盖全量
if missing_no:
    no_big_to_codes.setdefault(DEFAULT_BIG_CLASS, [])
    no_big_to_codes[DEFAULT_BIG_CLASS] = sorted(set(no_big_to_codes[DEFAULT_BIG_CLASS] + missing_no))

if missing_tc:
    tc_big_to_codes.setdefault(DEFAULT_BIG_CLASS, [])
    tc_big_to_codes[DEFAULT_BIG_CLASS] = sorted(set(tc_big_to_codes[DEFAULT_BIG_CLASS] + missing_tc))

print('\n[mapping] nombreOtorgante 大类(含Other) =', sorted(no_big_to_codes.keys()))
print('[mapping] tipoCredito 大类(含Other) =', sorted(tc_big_to_codes.keys()))

print('\n[校验] nombreOtorgante 未命中编码数量 =', len(missing_no))
if len(missing_no) <= 50:
    print('missing nombreOtorgante codes:', missing_no)

print('[校验] tipoCredito 未命中编码数量 =', len(missing_tc))
if len(missing_tc) <= 100:
    print('missing tipoCredito codes:', missing_tc)

print('\n[兜底] 未命中编码 ->', DEFAULT_BIG_CLASS)

cutoff_col = apply_time

[mapping] nombreOtorgante 大类(含Other) = ['Auto', 'Bank', 'Consumer', 'Education', 'FinanceCo', 'FundTrust', 'Government', 'Halo', 'Medical', 'MicroCredit', 'MultiPurposeFin', 'NonFinance', 'Other', 'OtherFinance', 'PersonalLoanCo', 'RealEstate', 'SIC', 'ServiceCredit']
[mapping] tipoCredito 大类(含Other) = ['Auto', 'Bond', 'Credit', 'CreditCard', 'Enterprise', 'FixedPay', 'GeneralLoan', 'GeneralPersonalLoan', 'Guarantee', 'Other', 'OtherLoan', 'PersonalLoan', 'RealEstate', 'UnsecuredCredit']

[校验] nombreOtorgante 未命中编码数量 = 6
missing nombreOtorgante codes: ['AGROPECUARIO', 'EDITORIAL', 'HIPOTECAGOBIERNO', 'REPORTE ESPECIAL', 'SOCIEDAD FINANCIERA COMUNITARIA', 'UNION DE CREDITO']
[校验] tipoCredito 未命中编码数量 = 1
missing tipoCredito codes: ['AV']

[兜底] 未命中编码 -> Other


In [3]:
import re

# =========================
# 查询板块特征衍生：核心逻辑（已做性能优化，避免逐列插入导致卡死）
# 目标：对每个 apply_id，在截止时间 cutoff_dt 之前的多个时间窗口内，
# 按 nombreOtorgante_big / tipoCredito_big 两套大类，计算：
# - 记录条数 cnt
# - 占比 ratio = cnt / 该窗口内总查询条数（分母为0则ratio=0）
# - importeCredito_num 的 7 个指标（std/mean/pos_rate/max/min/sum/notNull_rate）
# 缺失规则：
# - 若该大类在该窗口内 cnt=0：7个指标均为 -999
# - 若 cnt>0 但该大类所有记录 importeCredito_num 全缺失：7个指标均为 -999
# - 若部分缺失：缺失不参与 std/mean/max/min/sum/pos_rate；notNull_rate=nonnull/cnt
# 标准差口径：总体标准差（ddof=0）
# =========================

# 依赖列检查
req_cols = ['apply_id', 'fechaConsulta_dt', 'importeCredito_num', 'nombreOtorgante_big', 'tipoCredito_big']
missing_req = [c for c in req_cols if c not in consultas_flat.columns]
if missing_req:
    raise ValueError(f'missing required cols in consultas_flat: {missing_req}')

# apply_id 统一成字符串，避免 merge/index 不一致
consultas_flat['apply_id'] = consultas_flat['apply_id'].astype(str)
df_raw['apply_id'] = df_raw['apply_id'].astype(str)

# 合并 cutoff_dt（每条查询带上该客户的截止时间）
cutoff_df = df_raw[['apply_id', 'cutoff_dt']].copy()
consultas_enriched = consultas_flat.merge(cutoff_df, on='apply_id', how='left')

# 时间差（天）统一用时间戳（ns）计算
# diff_days = (cutoff_ns - consulta_ns) / (86400 * 1e9)
cutoff_ns = consultas_enriched['cutoff_dt'].values.astype('datetime64[ns]').astype('int64')
consulta_ns = consultas_enriched['fechaConsulta_dt'].values.astype('datetime64[ns]').astype('int64')
NAT_INT = np.iinfo(np.int64).min
mask_na = (cutoff_ns == NAT_INT) | (consulta_ns == NAT_INT)

diff_days = (cutoff_ns - consulta_ns).astype('float64') / (86400.0 * 1e9)
diff_days[mask_na] = np.nan
consultas_enriched['diff_days'] = diff_days

# 客户全集（保证窗口内无查询的客户也能产出特征）
all_apply_ids = df_raw['apply_id'].tolist()

# 大类列表：直接用映射大类（固定、英文），并包含 Other
no_classes = sorted(no_big_to_codes.keys())
tc_classes = sorted(tc_big_to_codes.keys())

# 列名安全化（避免空格/特殊符号）

def safe_token(x: str) -> str:
    x = str(x)
    return re.sub(r'[^A-Za-z0-9_]+', '', x)


def pivot_fill(series: pd.Series, index: list[str], columns: list[str], fill_value):
    """MultiIndex Series -> wide DataFrame，并补齐 index/columns"""
    wide = series.unstack(fill_value=fill_value)
    wide = wide.reindex(index=index, columns=columns, fill_value=fill_value)
    return wide


def to_wide(series: pd.Series, index: list[str], columns: list[str]):
    """MultiIndex Series -> wide DataFrame（保留 NaN，不填充）"""
    wide = series.unstack()
    wide = wide.reindex(index=index, columns=columns)
    return wide


def compute_amount_metrics(df_win: pd.DataFrame, cat_col: str, cat_values: list[str]):
    """计算金额相关 7 个指标，并按缺失规则输出（返回：metric->wide_df, wide_cnt）"""
    # cnt：包含缺失金额的记录
    cnt = df_win.groupby(['apply_id', cat_col]).size()

    g = df_win.groupby(['apply_id', cat_col])['importeCredito_num']
    nonnull = g.count()

    mean = g.mean()
    std0 = g.std(ddof=0)
    minv = g.min()
    maxv = g.max()
    sumv = g.sum(min_count=1)  # 全NaN -> NaN

    # 正值占比：只在非空金额上计算
    pos = np.where(df_win['importeCredito_num'].notna(), (df_win['importeCredito_num'] > 0).astype(int), np.nan)
    pos_rate = df_win.assign(_pos=pos).groupby(['apply_id', cat_col])['_pos'].mean()

    notnull_rate = nonnull / cnt

    wide_cnt = pivot_fill(cnt, index=all_apply_ids, columns=cat_values, fill_value=0)
    wide_nonnull = pivot_fill(nonnull, index=all_apply_ids, columns=cat_values, fill_value=0)

    # invalid：cnt==0 或 cnt>0但non-null==0
    invalid = (wide_cnt == 0) | ((wide_cnt > 0) & (wide_nonnull == 0))

    def finalize(wide: pd.DataFrame) -> pd.DataFrame:
        out = wide.copy()
        out[invalid] = -999
        return out

    out = {
        'amt_std': finalize(to_wide(std0, all_apply_ids, cat_values)),
        'amt_mean': finalize(to_wide(mean, all_apply_ids, cat_values)),
        'amt_pos_rate': finalize(to_wide(pos_rate, all_apply_ids, cat_values)),
        'amt_max': finalize(to_wide(maxv, all_apply_ids, cat_values)),
        'amt_min': finalize(to_wide(minv, all_apply_ids, cat_values)),
        'amt_sum': finalize(to_wide(sumv, all_apply_ids, cat_values)),
        'amt_notnull_rate': finalize(to_wide(notnull_rate, all_apply_ids, cat_values)),
    }
    return out, wide_cnt


# =========================
# 主流程：逐窗口产出特征
# =========================

feat_parts = []

for w in WINDOW_DAYS:
    # 过滤窗口：0 <= diff_days <= w
    m = consultas_enriched['diff_days'].between(0, w, inclusive='both')
    df_win = consultas_enriched.loc[
        m,
        ['apply_id', 'nombreOtorgante_code', 'nombreOtorgante_big', 'tipoCredito_big', 'importeCredito_num', 'diff_days'],
    ].copy()

    # 该窗口内总查询条数（分母），客户无记录则为0
    total_cnt = df_win.groupby('apply_id').size().reindex(all_apply_ids, fill_value=0)

    # 先写窗口级特征到 cols（避免后面忘了）
    cols = {}

    # 1) 每个客户该窗口总查询数
    cols[f'win_{w}d_total_cnt'] = total_cnt.astype('int32').to_numpy()

    # 2) 每个客户该窗口内 nombreOtorgante 唯一值个数（原始编码）
    no_nunique = df_win.groupby('apply_id')['nombreOtorgante_code'].nunique(dropna=True).reindex(all_apply_ids, fill_value=0)
    cols[f'win_{w}d_no_nunique'] = no_nunique.astype('int32').to_numpy()

    # 3) 每个客户该窗口内：所有查询记录的 importeCredito 统计（若总记录=0或有效金额=0，则7项均为-999）
    g_amt = df_win.groupby('apply_id')['importeCredito_num']
    amt_nonnull = g_amt.count().reindex(all_apply_ids, fill_value=0)
    invalid_amt = (total_cnt == 0) | ((total_cnt > 0) & (amt_nonnull == 0))

    amt_mean = g_amt.mean().reindex(all_apply_ids)
    amt_std = g_amt.std(ddof=0).reindex(all_apply_ids)
    amt_max = g_amt.max().reindex(all_apply_ids)
    amt_min = g_amt.min().reindex(all_apply_ids)
    try:
        amt_sum = g_amt.sum(min_count=1).reindex(all_apply_ids)
    except TypeError:
        # 兼容旧 pandas：没有 min_count 时，先 sum，再把 nonnull==0 的置 NaN
        amt_sum = g_amt.sum().reindex(all_apply_ids)
        amt_sum = amt_sum.where(amt_nonnull > 0, np.nan)

    # 正值占比：仅在非空金额上计算
    pos = np.where(df_win['importeCredito_num'].notna(), (df_win['importeCredito_num'] > 0).astype(int), np.nan)
    amt_pos_rate = df_win.assign(_pos=pos).groupby('apply_id')['_pos'].mean().reindex(all_apply_ids)

    amt_notnull_rate = (amt_nonnull / total_cnt.replace(0, np.nan)).reindex(all_apply_ids)

    def finalize_1d(s: pd.Series) -> pd.Series:
        out = s.copy()
        out[invalid_amt] = -999
        return out

    cols[f'win_{w}d_amt_std'] = finalize_1d(amt_std).astype('float32').to_numpy()
    cols[f'win_{w}d_amt_mean'] = finalize_1d(amt_mean).astype('float32').to_numpy()
    cols[f'win_{w}d_amt_pos_rate'] = finalize_1d(amt_pos_rate).astype('float32').to_numpy()
    cols[f'win_{w}d_amt_max'] = finalize_1d(amt_max).astype('float32').to_numpy()
    cols[f'win_{w}d_amt_min'] = finalize_1d(amt_min).astype('float32').to_numpy()
    cols[f'win_{w}d_amt_sum'] = finalize_1d(amt_sum).astype('float32').to_numpy()
    cols[f'win_{w}d_amt_notnull_rate'] = finalize_1d(amt_notnull_rate).astype('float32').to_numpy()

    # 4) 每个客户该窗口内：查询日期距离截止日期天数 max/min（无记录则 -999）
    g_diff = df_win.groupby('apply_id')['diff_days']
    diff_max = g_diff.max().reindex(all_apply_ids)
    diff_min = g_diff.min().reindex(all_apply_ids)
    invalid_diff = (total_cnt == 0)

    diff_max = diff_max.where(~invalid_diff, -999)
    diff_min = diff_min.where(~invalid_diff, -999)

    cols[f'win_{w}d_diff_max'] = diff_max.astype('float32').to_numpy()
    cols[f'win_{w}d_diff_min'] = diff_min.astype('float32').to_numpy()

    # 5) 平均查询数：总查询数 / 查询日期距离截止日期天数
    # 这里用 diff_max 作为“覆盖的最大天数跨度”；避免除0，denom<=0 时按 1 处理。
    denom = np.maximum(diff_max.replace(-999, np.nan).fillna(0).to_numpy(), 1.0)
    q_per_day = (total_cnt.astype('float64').to_numpy() / denom).astype('float32')
    q_per_day[invalid_diff.to_numpy()] = -999
    cols[f'win_{w}d_q_per_day'] = q_per_day

    # ---------- nombreOtorgante_big ----------
    no_cnt = df_win.groupby(['apply_id', 'nombreOtorgante_big']).size()
    no_cnt_wide = pivot_fill(no_cnt, index=all_apply_ids, columns=no_classes, fill_value=0)
    no_ratio_wide = no_cnt_wide.div(total_cnt.replace(0, np.nan), axis=0).fillna(0.0)
    no_amt_dict, _ = compute_amount_metrics(df_win, 'nombreOtorgante_big', no_classes)

    # ---------- tipoCredito_big ----------
    tc_cnt = df_win.groupby(['apply_id', 'tipoCredito_big']).size()
    tc_cnt_wide = pivot_fill(tc_cnt, index=all_apply_ids, columns=tc_classes, fill_value=0)
    tc_ratio_wide = tc_cnt_wide.div(total_cnt.replace(0, np.nan), axis=0).fillna(0.0)
    tc_amt_dict, _ = compute_amount_metrics(df_win, 'tipoCredito_big', tc_classes)

    # 继续往 cols 里追加大类特征列（cols 已包含窗口级特征）

    # nombreOtorgante features
    for cls in no_classes:
        c = safe_token(cls)
        cols[f'no_{c}_{w}d_cnt'] = no_cnt_wide[cls].astype('int32').to_numpy()
        cols[f'no_{c}_{w}d_ratio'] = no_ratio_wide[cls].astype('float32').to_numpy()

    for metric_name, wide in no_amt_dict.items():
        for cls in no_classes:
            c = safe_token(cls)
            cols[f'no_{c}_{w}d_{metric_name}'] = wide[cls].astype('float32').to_numpy()

    # tipoCredito features
    for cls in tc_classes:
        c = safe_token(cls)
        cols[f'tc_{c}_{w}d_cnt'] = tc_cnt_wide[cls].astype('int32').to_numpy()
        cols[f'tc_{c}_{w}d_ratio'] = tc_ratio_wide[cls].astype('float32').to_numpy()

    for metric_name, wide in tc_amt_dict.items():
        for cls in tc_classes:
            c = safe_token(cls)
            cols[f'tc_{c}_{w}d_{metric_name}'] = wide[cls].astype('float32').to_numpy()

    part = pd.DataFrame(cols, index=all_apply_ids)
    feat_parts.append(part)

# 合并所有窗口特征
consultas_features = pd.concat(feat_parts, axis=1)
consultas_features.insert(0, 'apply_id', consultas_features.index)

# 添加cust_no列（从df_raw中获取）
cust_no_map = df_raw.set_index('apply_id')['cust_no']
consultas_features.insert(1, 'cust_no', consultas_features['apply_id'].map(cust_no_map))

# 添加create_time列（从df_raw中获取apply_time）
create_time_map = df_raw.set_index('apply_id')['apply_time']
consultas_features.insert(2, 'create_time', consultas_features['apply_id'].map(create_time_map))


print('consultas_features.shape =', consultas_features.shape)
consultas_features.head(3)

consultas_features.shape = (14963, 2103)


,apply_id,cust_no,create_time,win_30d_total_cnt,win_30d_no_nunique,win_30d_amt_std,win_30d_amt_mean,win_30d_amt_pos_rate,win_30d_amt_max,win_30d_amt_min,...,tc_Enterprise_36000d_amt_notnull_rate,tc_FixedPay_36000d_amt_notnull_rate,tc_GeneralLoan_36000d_amt_notnull_rate,tc_GeneralPersonalLoan_36000d_amt_notnull_rate,tc_Guarantee_36000d_amt_notnull_rate,tc_Other_36000d_amt_notnull_rate,tc_OtherLoan_36000d_amt_notnull_rate,tc_PersonalLoan_36000d_amt_notnull_rate,tc_RealEstate_36000d_amt_notnull_rate,tc_UnsecuredCredit_36000d_amt_notnull_rate
1355047166686003201,1355047166686003201,800001818243,2026-03-02 10:44:36.872,4,4,0.000000,1000.000000,1.000000,1000.0,1000.0,...,-999.0,1.0,1.0,1.0,-999.0,-999.0,-999.0,-999.0,-999.0,-999.0
1355151139254296577,1355151139254296577,800001804693,2026-03-02 11:35:04.922,7,4,349.927094,857.142883,0.857143,1000.0,0.0,...,-999.0,1.0,1.0,1.0,-999.0,-999.0,-999.0,-999.0,-999.0,-999.0
1355334860775358465,1355334860775358465,800001818116,2026-03-02 13:04:13.012,1,1,0.000000,1000.000000,1.000000,1000.0,1000.0,...,-999.0,1.0,1.0,1.0,-999.0,-999.0,-999.0,1.0,-999.0,-999.0


In [4]:
# =========================
# 查询板块特征衍生：输出特征字典 + 导出结果 CSV
# - 特征名：cdc_{原特征名}_v3
# - 特征字典三列：特征名称 / 中文释义(全中文) / 逻辑解释(全中文)
# - 结果：consultas_features 导出 csv
# =========================

from pathlib import Path
import re
import pandas as pd
import numpy as np

out_dir = Path('outputs')
out_dir.mkdir(parents=True, exist_ok=True)

# 1) 导出衍生结果
feat_out = consultas_features.copy()
result_csv_path = out_dir / 'consultas_features_cdc_v3_up.csv'
feat_out.to_csv(result_csv_path, index=False, encoding='utf-8-sig')
print('已导出衍生特征结果CSV:', result_csv_path, 'shape=', feat_out.shape)

# 2) 生成特征字典
# 大类中文释义（用于字典，避免中文出现在特征名里，中文只出现在释义/逻辑解释）
NO_BIG_CN = {
    'SIC': '征信/信用信息机构',
    'PersonalLoanCo': '个人贷款公司',
    'MultiPurposeFin': '多用途金融公司（SOFOM）',
    'MicroCredit': '小额信贷机构',
    'ServiceCredit': '服务类授信机构',
    'FinanceCo': '金融公司/普惠金融机构',
    'Bank': '银行',
    'Consumer': '消费/零售类商户',
    'Auto': '汽车/车辆相关机构',
    'FundTrust': '基金与信托机构',
    'RealEstate': '房地产/不动产机构',
    'NonFinance': '非金融机构/非金融租赁类',
    'OtherFinance': '其他金融机构',
    'Medical': '医疗健康机构',
    'Government': '政府机构',
    'Education': '教育机构',
    'Halo': '其他/特殊类别',
    'Other': '其他（未映射）',
}

TC_BIG_CN = {
    'Auto': '汽车贷款/车贷类',
    'Enterprise': '企业贷款类',
    'OtherLoan': '其他贷款类',
    'Credit': '信用/循环授信类',
    'CreditCard': '信用卡类',
    'RealEstate': '房贷/不动产抵押类',
    'Bond': '债券/证券类',
    'Guarantee': '担保/保证类',
    'GeneralPersonalLoan': '通用个人贷款类',
    'FixedPay': '固定分期类',
    'GeneralLoan': '通用贷款类',
    'PersonalLoan': '个人贷款类',
    'UnsecuredCredit': '无担保授信类',
    'Other': '其他（未映射）',
}

FIELD_CN = {
    'cutoff_dt': '截止日期',
    'request_time': '截止日期（请求时间）',
    'apply_time': '截止日期（申请时间）',
    'fechaConsulta': '查询日期',
    'nombreOtorgante': '机构类别',
    'tipoCredito': '信贷/贷款类型',
    'importeCredito': '授信/贷款金额（合同金额）',
}

METRIC_CN = {
    'total_cnt': '查询记录数',
    'no_nunique': '机构类别唯一值个数',
    'amt_std': '金额标准差',
    'amt_mean': '金额平均值',
    'amt_pos_rate': '金额正值占比',
    'amt_max': '金额最大值',
    'amt_min': '金额最小值',
    'amt_sum': '金额总和',
    'amt_notnull_rate': '金额有效值占比',
    'diff_max': '距截止日期最大天数',
    'diff_min': '距截止日期最小天数',
    'q_per_day': '日均查询次数',
    'cnt': '数量',
    'ratio': '占比',
}

AMT_METRICS_CN = {
    'amt_std': '金额标准差',
    'amt_mean': '金额平均值',
    'amt_pos_rate': '金额正值占比',
    'amt_max': '金额最大值',
    'amt_min': '金额最小值',
    'amt_sum': '金额总和',
    'amt_notnull_rate': '金额有效值占比',
}


def window_cn(w: int) -> str:
    if int(w) >= 36000:
        return '全量历史（近似）'
    return f'近{int(w)}天'


def base_logic_window(w: int) -> str:
    # cutoff_col 在第2格已定义
    cutoff_cn = FIELD_CN.get(cutoff_col, '截止日期')
    return (
        f"以{cutoff_col}({cutoff_cn})为截止日期，与fechaConsulta({FIELD_CN['fechaConsulta']})计算diff_days=截止日期-查询日期(天)，"
        f"筛选diff_days在[0,{w}]内的查询记录（{window_cn(w)}）后进行统计；"
        f"金额类统计基于importeCredito({FIELD_CN['importeCredito']})；当窗口内无记录或窗口内有记录但金额全为空时，金额相关指标填-999。"
    )


def mk_row(col: str):
    # 返回：特征名(带cdc_/_v3) + 全中文释义 + 全中文逻辑解释
    if col == 'apply_id':
        return None

    feat_name = f'cdc_{col}_v3'

    # 1) 窗口级
    m = re.match(r'^win_(\d+)d_(.+)$', col)
    if m:
        w = int(m.group(1))
        metric = m.group(2)
        metric_cn = METRIC_CN.get(metric, metric)
        cn_desc = f"{window_cn(w)}查询板块：{metric_cn}"

        if metric == 'total_cnt':
            logic = base_logic_window(w) + ' 统计窗口内查询记录条数。'
        elif metric == 'no_nunique':
            logic = base_logic_window(w) + f" 统计窗口内nombreOtorgante({FIELD_CN['nombreOtorgante']})的去重个数。"
        elif metric.startswith('amt_'):
            logic = base_logic_window(w) + f" 对窗口内全部记录的importeCredito({FIELD_CN['importeCredito']})计算{METRIC_CN.get(metric, metric)}。"
        elif metric in ('diff_max', 'diff_min'):
            logic = base_logic_window(w) + f" 对窗口内diff_days计算{METRIC_CN.get(metric, metric)}。"
        elif metric == 'q_per_day':
            logic = base_logic_window(w) + " 计算日均查询次数=窗口内查询记录数/窗口内距截止日期最大天数(diff_max)，若无记录则填-999。"
        else:
            logic = base_logic_window(w) + ' 计算窗口级统计指标。'

        return feat_name, cn_desc, logic

    # 2) nombreOtorgante 大类
    m = re.match(r'^no_([^_]+)_(\d+)d_(cnt|ratio|amt_.+)$', col)
    if m:
        cls = m.group(1)
        w = int(m.group(2))
        metric = m.group(3)
        cls_cn = NO_BIG_CN.get(cls, '其他（未映射）')
        metric_cn = METRIC_CN.get(metric, metric)

        if metric == 'cnt':
            cn_desc = f"{window_cn(w)}查询板块：机构大类[{cls_cn}]数量"
            logic = base_logic_window(w) + f" 将nombreOtorgante({FIELD_CN['nombreOtorgante']})映射为机构大类后，统计窗口内机构大类={cls}({cls_cn})的记录条数。"
        elif metric == 'ratio':
            cn_desc = f"{window_cn(w)}查询板块：机构大类[{cls_cn}]占比"
            logic = base_logic_window(w) + f" 机构大类={cls}({cls_cn})的数量/窗口内查询记录总数(total_cnt)，若total_cnt=0则占比记为0。"
        elif metric in AMT_METRICS_CN:
            cn_desc = f"{window_cn(w)}查询板块：机构大类[{cls_cn}]{AMT_METRICS_CN[metric]}"
            logic = base_logic_window(w) + (
                f" 在机构大类={cls}({cls_cn})的子集内，对importeCredito({FIELD_CN['importeCredito']})计算{AMT_METRICS_CN[metric]}；"
                f"标准差使用总体标准差(ddof=0)。"
            )
        else:
            cn_desc = f"{window_cn(w)}查询板块：机构大类[{cls_cn}]{metric_cn}"
            logic = base_logic_window(w) + f" 在机构大类={cls}({cls_cn})的子集内计算{metric_cn}。"

        return feat_name, cn_desc, logic

    # 3) tipoCredito 大类
    m = re.match(r'^tc_([^_]+)_(\d+)d_(cnt|ratio|amt_.+)$', col)
    if m:
        cls = m.group(1)
        w = int(m.group(2))
        metric = m.group(3)
        cls_cn = TC_BIG_CN.get(cls, '其他（未映射）')
        metric_cn = METRIC_CN.get(metric, metric)

        if metric == 'cnt':
            cn_desc = f"{window_cn(w)}查询板块：信贷类型大类[{cls_cn}]数量"
            logic = base_logic_window(w) + f" 将tipoCredito({FIELD_CN['tipoCredito']})映射为信贷类型大类后，统计窗口内信贷类型大类={cls}({cls_cn})的记录条数。"
        elif metric == 'ratio':
            cn_desc = f"{window_cn(w)}查询板块：信贷类型大类[{cls_cn}]占比"
            logic = base_logic_window(w) + f" 信贷类型大类={cls}({cls_cn})的数量/窗口内查询记录总数(total_cnt)，若total_cnt=0则占比记为0。"
        elif metric in AMT_METRICS_CN:
            cn_desc = f"{window_cn(w)}查询板块：信贷类型大类[{cls_cn}]{AMT_METRICS_CN[metric]}"
            logic = base_logic_window(w) + (
                f" 在信贷类型大类={cls}({cls_cn})的子集内，对importeCredito({FIELD_CN['importeCredito']})计算{AMT_METRICS_CN[metric]}；"
                f"标准差使用总体标准差(ddof=0)。"
            )
        else:
            cn_desc = f"{window_cn(w)}查询板块：信贷类型大类[{cls_cn}]{metric_cn}"
            logic = base_logic_window(w) + f" 在信贷类型大类={cls}({cls_cn})的子集内计算{metric_cn}。"

        return feat_name, cn_desc, logic

    # 4) 兜底：仍然保证中文释义/逻辑可用
    cn_desc = '查询板块：特征（未识别命名模式）'
    logic = '该特征未匹配到预设命名模式，请人工检查其命名与生成逻辑。'
    return feat_name, cn_desc, logic


rows = []
for c in feat_out.columns:
    r = mk_row(c)
    if r is not None:
        rows.append(r)

feature_dict = pd.DataFrame(rows, columns=['feature_name', 'cn_desc', 'logic_cn'])

# =========================
# 3) 追加字典统计列：空值率/唯一值个数/IV（全样本 + 两个子样本）
# - 空值判定：NaN 或 -999
# - IV：y=fpd7（二分类0/1），缺失y不参与
# - 子样本：approve_state == 'CYCLE_PASS' / 'SINGLE_PASS'
# =========================
label_df = df_raw[['apply_id', 'fpd7', 'approve_state']].copy()
label_df['apply_id'] = label_df['apply_id'].astype(str)

work = feat_out.merge(label_df, on='apply_id', how='left')

y = pd.to_numeric(work['fpd7'], errors='coerce')
# 仅保留 0/1，其它值视为缺失
mask_y_valid = y.isin([0, 1])
work = work.loc[mask_y_valid].copy()
y = y.loc[mask_y_valid].astype(int)

mask_cycle = (work['approve_state'] == 'CYCLE_PASS')
mask_single = (work['approve_state'] == 'SINGLE_PASS')


def calc_iv(x: pd.Series, y: pd.Series, max_bins: int = 10, max_cat: int = 50) -> float:
    """计算IV（Information Value）。

    - x 的缺失（NaN 或 -999）单独作为一个箱
    - 数值型：对非缺失部分做分箱（qcut）
    - 类别型：高基数时保留topN，其余合并为Other
    """
    if x is None or len(x) == 0:
        return np.nan

    # y 必须同时包含 0 和 1，否则IV无意义
    yv = y.values
    if (yv == 1).sum() == 0 or (yv == 0).sum() == 0:
        return np.nan

    x0 = x.copy()
    miss = x0.isna() | (x0 == -999)

    # 判断是否数值型
    is_num = pd.api.types.is_numeric_dtype(x0)

    if is_num:
        xn = pd.to_numeric(x0, errors='coerce')
        xn_miss = miss | xn.isna()
        x_non = xn[~xn_miss]

        if x_non.nunique(dropna=True) <= 1:
            # 只有一个有效取值，直接按取值+缺失分组
            b = xn.astype('object')
            b[xn_miss] = '__MISSING__'
        else:
            # qcut 分箱
            try:
                b_non = pd.qcut(x_non, q=min(max_bins, x_non.nunique()), duplicates='drop')
            except Exception:
                # 回退：按原值当类别
                b_non = x_non.astype('object')
            b = pd.Series(index=xn.index, dtype='object')
            b.loc[x_non.index] = b_non.astype('object').astype(str)
            b.loc[xn_miss] = '__MISSING__'
    else:
        xs = x0.astype('object')
        xs = xs.where(~miss, '__MISSING__')
        # 合并长尾类别
        vc = xs.value_counts(dropna=False)
        keep = set(vc.head(max_cat).index.tolist())
        b = xs.where(xs.isin(keep), 'Other')

    # 计算woe/iv
    df_tmp = pd.DataFrame({'b': b, 'y': y})
    grp = df_tmp.groupby('b')['y']
    bad = grp.sum()  # y==1
    cnt = grp.count()
    good = cnt - bad

    bad_total = bad.sum()
    good_total = good.sum()
    if bad_total == 0 or good_total == 0:
        return np.nan

    # 平滑，避免除0/ln(0)
    eps = 1e-6
    dist_bad = (bad / bad_total).clip(lower=eps)
    dist_good = (good / good_total).clip(lower=eps)
    woe = np.log(dist_bad / dist_good)
    iv = ((dist_bad - dist_good) * woe).sum()
    return float(iv)


feat_cols = [c for c in feat_out.columns if c != 'apply_id']

missing_rate_list = []
nunique_list = []
iv_all_list = []
iv_cycle_list = []
iv_single_list = []

for i, c in enumerate(feat_cols, 1):
    x = work[c]
    miss = x.isna() | (x == -999)
    missing_rate_list.append(float(miss.mean()))
    nunique_list.append(int(x.mask(x == -999).nunique(dropna=True)))

    iv_all_list.append(calc_iv(x, y))
    iv_cycle_list.append(calc_iv(x[mask_cycle], y[mask_cycle]))
    iv_single_list.append(calc_iv(x[mask_single], y[mask_single]))

    if i % 200 == 0:
        print(f'IV进度: {i}/{len(feat_cols)}')

feature_dict['missing_rate'] = missing_rate_list
feature_dict['nunique'] = nunique_list
feature_dict['iv_fpd7'] = iv_all_list
feature_dict['iv_cycle_pass'] = iv_cycle_list
feature_dict['iv_single_pass'] = iv_single_list

# IV 保留4位小数
for c in ['iv_fpd7', 'iv_cycle_pass', 'iv_single_pass']:
    feature_dict[c] = pd.to_numeric(feature_dict[c], errors='coerce').round(4)

# 按 iv_cycle_pass 从大到小排序（缺失放最后）
feature_dict = feature_dict.sort_values(
    by='iv_cycle_pass', ascending=False, na_position='last', kind='mergesort'
).reset_index(drop=True)

# =========================
# 4) 保存字典
# =========================
dict_csv_path = out_dir / 'consultas_features_dict_cdc_v3_up.csv'
feature_dict.to_csv(dict_csv_path, index=False, encoding='utf-8-sig')
print('已导出特征字典CSV:', dict_csv_path, 'rows=', len(feature_dict))

feature_dict.head(20)

已导出衍生特征结果CSV: outputs/consultas_features_cdc_v3_up.csv shape= (14963, 2103)
IV进度: 200/2102
IV进度: 400/2102
IV进度: 600/2102
IV进度: 800/2102
IV进度: 1000/2102
IV进度: 1200/2102
IV进度: 1400/2102
IV进度: 1600/2102
IV进度: 1800/2102
IV进度: 2000/2102
已导出特征字典CSV: outputs/consultas_features_dict_cdc_v3_up.csv rows= 2102


,feature_name,cn_desc,logic_cn,missing_rate,nunique,iv_fpd7,iv_cycle_pass,iv_single_pass
0,cdc_cust_no_v3,查询板块：特征（未识别命名模式）,该特征未匹配到预设命名模式，请人工检查其命名与生成逻辑。,NaN,0,NaN,NaN,NaN
1,cdc_create_time_v3,查询板块：特征（未识别命名模式）,该特征未匹配到预设命名模式，请人工检查其命名与生成逻辑。,NaN,0,NaN,NaN,NaN
2,cdc_win_30d_total_cnt_v3,近30天查询板块：查询记录数,以apply_time(截止日期（申请时间）)为截止日期，与fechaConsulta(查询...,NaN,0,NaN,NaN,NaN
3,cdc_win_30d_no_nunique_v3,近30天查询板块：机构类别唯一值个数,以apply_time(截止日期（申请时间）)为截止日期，与fechaConsulta(查询...,NaN,0,NaN,NaN,NaN
4,cdc_win_30d_amt_std_v3,近30天查询板块：金额标准差,以apply_time(截止日期（申请时间）)为截止日期，与fechaConsulta(查询...,NaN,0,NaN,NaN,NaN
5,cdc_win_30d_amt_mean_v3,近30天查询板块：金额平均值,以apply_time(截止日期（申请时间）)为截止日期，与fechaConsulta(查询...,NaN,0,NaN,NaN,NaN
6,cdc_win_30d_amt_pos_rate_v3,近30天查询板块：金额正值占比,以apply_time(截止日期（申请时间）)为截止日期，与fechaConsulta(查询...,NaN,0,NaN,NaN,NaN
7,cdc_win_30d_amt_max_v3,近30天查询板块：金额最大值,以apply_time(截止日期（申请时间）)为截止日期，与fechaConsulta(查询...,NaN,0,NaN,NaN,NaN
8,cdc_win_30d_amt_min_v3,近30天查询板块：金额最小值,以apply_time(截止日期（申请时间）)为截止日期，与fechaConsulta(查询...,NaN,0,NaN,NaN,NaN
9,cdc_win_30d_amt_sum_v3,近30天查询板块：金额总和,以apply_time(截止日期（申请时间）)为截止日期，与fechaConsulta(查询...,NaN,0,NaN,NaN,NaN


## 信用板块衍生


In [5]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

# =========================
# 信用板块（creditos）衍生：前提准备
# - 时间窗口：30/60/90/180/360/720/36000（36000 近似全量）
# - 截止日期字段：优先 request_time，否则 apply_time
# - 时间窗口判断：用时间戳(ns)差值换算成天数 diff_days，再判断是否落在窗口内
# - 映射：tipoResponsabilidad / frecuenciaPagos / tipoCredito / nombreOtorgante
#   未命中统一归到“其他”
# =========================

WINDOW_DAYS = [30, 60, 90, 180, 360, 720, 36000]
DEFAULT_OTHER = 'Other'

# 截止时间字段：优先 request_time，否则 apply_time
cutoff_col = 'request_time' if 'request_time' in df_raw.columns else 'apply_time'
df_raw['cutoff_dt'] = pd.to_datetime(df_raw[cutoff_col], errors='coerce')
print('cutoff_col =', cutoff_col)


def parse_response_body(x):
    """将 response_body 解析为 dict（兼容双层json）。"""
    if x is None:
        return {}
    if isinstance(x, dict):
        return x
    if not isinstance(x, str):
        x = str(x)
    x = x.strip()
    if not x:
        return {}
    obj = json.loads(x)
    if isinstance(obj, str):
        obj = json.loads(obj)
    return obj if isinstance(obj, dict) else {}


# =========================
# 1) creditos 平铺
# =========================
creditos_recs = []
for row in df_raw.itertuples(index=False):
    d = row._asdict()
    obj = parse_response_body(d.get('response_body'))
    creditos = obj.get('creditos', []) or []
    for c in creditos:
        if not isinstance(c, dict):
            continue
        creditos_recs.append({
            'apply_id': str(d.get('apply_id')),
            'cutoff_dt': d.get('cutoff_dt'),
            # creditos 常用日期字段（不同版本可能命名略有差异）
            'fechaActualizacion': c.get('fechaActualizacion'),
            'fechaReporte': c.get('fechaReporte'),
            'fechaAperturaCuenta': c.get('fechaAperturaCuenta'),
            'fechaUltimaCompra': c.get('fechaUltimaCompra'),
            'fechaCierreCuenta': c.get('fechaCierreCuenta'),
            # 本次准备需要的字段（类别）
            'tipoResponsabilidad': c.get('tipoResponsabilidad'),
            'frecuenciaPagos': c.get('frecuenciaPagos'),
            'tipoCuenta': c.get('tipoCuenta'),
            'tipoCredito': c.get('tipoCredito'),
            'nombreOtorgante': c.get('nombreOtorgante'),
            # 本次35指标需要的字段（数值）
            'saldoVencido': c.get('saldoVencido'),
            'limiteCredito': c.get('limiteCredito'),
            'saldoActual': c.get('saldoActual'),
            'montoPagar': c.get('montoPagar'),
            'valorActivoValuacion': c.get('valorActivoValuacion'),
            'numeroPagos': c.get('numeroPagos'),
            'peorAtraso': c.get('peorAtraso'),
        })

creditos_flat = pd.DataFrame.from_records(creditos_recs)

# 类型清洗
for col in ['tipoResponsabilidad', 'frecuenciaPagos', 'tipoCuenta', 'tipoCredito', 'nombreOtorgante']:
    creditos_flat[col] = creditos_flat[col].astype(str).str.strip().replace({'None': np.nan, 'nan': np.nan})

creditos_flat['fechaActualizacion_dt'] = pd.to_datetime(creditos_flat['fechaActualizacion'], errors='coerce')
creditos_flat['fechaReporte_dt'] = pd.to_datetime(creditos_flat['fechaReporte'], errors='coerce')
creditos_flat['fechaAperturaCuenta_dt'] = pd.to_datetime(creditos_flat['fechaAperturaCuenta'], errors='coerce')
creditos_flat['fechaUltimaCompra_dt'] = pd.to_datetime(creditos_flat['fechaUltimaCompra'], errors='coerce')
creditos_flat['fechaCierreCuenta_dt'] = pd.to_datetime(creditos_flat['fechaCierreCuenta'], errors='coerce')

print('creditos_flat.shape =', creditos_flat.shape)

# =========================
# 你要求新增 3 套“窗口计算日期”
# - upd: fechaActualizacion（更新日期，缺失用 fechaReporte 兜底）
# - open: fechaAperturaCuenta（账户开立日期）
# - lastbuy: fechaUltimaCompra（最后消费日期）
# 统一用 ns 时间戳差 -> 天数 diff_days_*
# =========================

def compute_diff_days(event_dt: pd.Series) -> np.ndarray:
    cutoff_ns = creditos_flat['cutoff_dt'].values.astype('datetime64[ns]').astype('int64')
    event_ns = event_dt.values.astype('datetime64[ns]').astype('int64')
    NAT_INT = np.iinfo(np.int64).min
    mask_na = (cutoff_ns == NAT_INT) | (event_ns == NAT_INT)
    diff = (cutoff_ns - event_ns).astype('float64') / (86400.0 * 1e9)
    diff[mask_na] = np.nan
    return diff

creditos_flat['event_dt_upd'] = creditos_flat['fechaActualizacion_dt'].combine_first(creditos_flat['fechaReporte_dt'])
creditos_flat['event_dt_open'] = creditos_flat['fechaAperturaCuenta_dt']
creditos_flat['event_dt_lastbuy'] = creditos_flat['fechaUltimaCompra_dt']

creditos_flat['diff_days_upd'] = compute_diff_days(creditos_flat['event_dt_upd'])
creditos_flat['diff_days_open'] = compute_diff_days(creditos_flat['event_dt_open'])
creditos_flat['diff_days_lastbuy'] = compute_diff_days(creditos_flat['event_dt_lastbuy'])

print('event(upd)=fechaActualizacion_dt优先，兜底fechaReporte_dt；缺失率=', float(creditos_flat['event_dt_upd'].isna().mean()))
print('event(open)=fechaAperturaCuenta_dt；缺失率=', float(creditos_flat['event_dt_open'].isna().mean()))
print('event(lastbuy)=fechaUltimaCompra_dt；缺失率=', float(creditos_flat['event_dt_lastbuy'].isna().mean()))

# =========================
# 2) 取值归类（未命中 -> 其他）
# =========================

# tipoResponsabilidad 责任类型（英文大类名，用于生成特征名；中文释义后续写入特征字典）
resp_to_big = {
    'I': 'Individual',        # 个人/单独责任（主贷人/持有人）
    'O': 'SolidaryObligor',   # 连带责任人/连带债务人（共同连带）
    'T': 'HolderAval',        # 主贷人 + 担保人（主贷人并有担保）
    'M': 'Joint',             # 联名/共同责任（joint）
    'A': 'Guarantor',         # 担保人/保证人
}

# frecuenciaPagos 还款频率（英文大类名）
freq_to_big = {
    'M': 'Monthly',          # 月付
    'S': 'Weekly',           # 周付
    'Q': 'BiWeekly',         # 半月/每两周
    'R': 'RevolvingMin',     # 循环账户最低还款
    'U': 'OneTime',          # 一次性/单期
    'C': 'Other',            # 其他
    'B': 'Other',
    'T': 'Other',
    'A': 'Other',
    'D': 'Other',
    'E': 'Other',
}

# tipoCuenta 账户类型（英文大类名；这是你指出的 F/R/P/L 等）
# 注意：本次35指标里暂时不按 tipoCuenta 分组，先在此保留映射，后续需要时可直接用。
ta_to_big = {
    'F': 'FixedPay',   # 固定分期/固定还款账户
    'R': 'Revolving',  # 循环账户（Revolvente）
    'P': 'Pledge',     # 质押贷款（Prendario）
    'L': 'NoLimit',    # 无预设额度账户
    'H': 'Other',
    'Q': 'Other',
    'A': 'Other',
    'E': 'Other',
}

# tipoCredito 信用/贷款类型（英文大类名；按你给的 PP/CC/LC/... 归类）
# 未命中统一归到 Other

tc_to_big = {
    # 个人贷款
    'PP': 'PersonalLoan',

    # 消费信贷
    'CC': 'ConsumerCredit',

    # 授信额度/信用额度
    'LC': 'CreditLine',

    # 信用卡
    'TC': 'CreditCard',

    # 个人消费贷
    'CP': 'ConsumerLoan',

    # 工资贷/代发工资贷款
    'PN': 'PayrollLoan',

    # 联保小组/互助小组贷款
    'GS': 'GroupLoan',

    # 家电/家具分期
    'AM': 'Appliance',

    # 房产
    'HV': 'RealEstate',
    'BR': 'RealEstate',
    'MC': 'RealEstate',

    # 担保
    'TG': 'Guarantee',
    'PG': 'Guarantee',
    'FT': 'Guarantee',
    'FI': 'Guarantee',

    # 车
    'CA': 'Auto',
    'AA': 'Auto',
    'AB': 'Auto',

    # 企业
    'AE': 'Enterprise',
    'PM': 'Enterprise',

    # 其他
    'NC': 'Other',
    'PE': 'Other',
    'PB': 'Other',
    'TS': 'Other',
    'TD': 'Other',
    'NG': 'Other',
    'CO': 'Other',
    'EQ': 'Other',
    'OT': 'Other',
    'CF': 'Other',
    'LR': 'Other',
    'AR': 'Other',
    'BC': 'Other',
    'PQ': 'Other',
}

# nombreOtorgante 机构类别（英文大类名）
no_to_big = {
    # 个人贷款公司信息
    'COMPANIA DE PRESTAMO PERSONAL': 'PersonalLoanCo',
    'SOFOL PRESTAMO PERSONAL': 'PersonalLoanCo',
    'SOFOL EMPRESARIAL': 'PersonalLoanCo',
    'SOFOL HIPOTECARIA': 'PersonalLoanCo',
    'SOFOL AUTOMOTRIZ': 'PersonalLoanCo',

    # 多用途金融公司信息
    'SOCIEDAD FINANCIERA DE OBJETO MULTIPLE': 'MultiPurposeFin',

    # 小额信贷信息
    'MICROFINANCIERA': 'MicroCredit',
    'MIC CREDITO PERS': 'MicroCredit',
    'CASA DE EMPENO': 'MicroCredit',

    # 百货信息
    'TIENDA DEPARTAMENTAL': 'Retail',
    'TIENDA COMERCIAL': 'Retail',
    'VENTA POR CATALOGO': 'Retail',
    'TIENDA DE ROPA': 'Retail',
    'TIENDA DE AUTOSERVICIO': 'Retail',
    'VENTA POR CORREO / TELEFONO': 'Retail',
    'MERCANCIA PARA HOGAR Y OFICINA': 'Retail',
    'MERCANCIA PARA LA CONSTRUCCION': 'Retail',

    # 金融公司信息
    'FINANCIERA': 'FinanceCo',
    'SOCIEDADES FINANCIERAS POPULARES': 'FinanceCo',
    'ARRENDADORAS FINANCIERAS': 'FinanceCo',
    'SOCIEDAD FINANCIERA COMUNITARIA': 'FinanceCo',
    'HIPOTECARIA': 'FinanceCo',
    'CIA Q  OTORGA': 'FinanceCo',
    'SEGURO': 'FinanceCo',

    # 银行信息
    'BANCOS': 'Bank',
    'BANCO': 'Bank',

    # 服务
    'SERVICIOS': 'Service',
    'SERVS. GRALES.': 'Service',

    # 车
    'AUTOMOTRIZ': 'Auto',
    'COMPANIA DE FINANCIAMIENTO AUTOMOTRIZ': 'Auto',
    'COMPANIA DE FINANCIAMIENTO DE MOTOCICLET': 'Auto',
    'OTROS VEHICULOS': 'Auto',

    # SIC
    'SIC': 'SIC',

    # 房产
    'HIPOTECAGOBIERNO': 'RealEstate',
    'BIENES RAICES': 'RealEstate',

    # 政府
    'GUBERNAMENTALES': 'Government',
    'GOBIERNO': 'Government',

    # 教育
    'EDUCACION': 'Education',

    # 医疗
    'SALUD Y SERVICIOS MEDICOS': 'Medical',

    # 其他金融
    'ADMINISTRADORAS DE CARTERA': 'OtherFinance',
    'HIPOTECARIO NO BANCARIO': 'OtherFinance',
    'CAJAS DE AHORRO': 'OtherFinance',
    'COOPERATIVA': 'OtherFinance',
    'COOPERATIVA DE AHORRO Y CREDITO': 'OtherFinance',
    'UNION DE CREDITO': 'OtherFinance',
    'FACTORAJE': 'OtherFinance',
    'AUTOFINANCIAMIENTO': 'OtherFinance',
    'COBRANZA': 'OtherFinance',
    'OTRAS FINANCIERA': 'OtherFinance',
    'ARRENDAMIENTO': 'OtherFinance',
    'ARRENDADORA': 'OtherFinance',

    # 媒体
    'COMUNICACIONES': 'Media',
    'TELEFONIA CELULAR': 'Media',
    'SERVICIO DE TELEVISION DE PAGA': 'Media',
    'EDITORIAL': 'Media',
    'TELEFONIA LOCAL Y DE LARGA DISTANCIA': 'Media',

    # 非金融
    'ARRENDADORAS NO FINANCIERAS': 'NonFinance',

    # 基金与信托
    'FONDOS Y FIDEICOMISOS': 'FundTrust',
    'FONDOS Y FIDEIC': 'FundTrust',
    'FONDOS Y FIDEICO': 'FundTrust',
}

# 标准化（大写）
creditos_flat['resp_code'] = creditos_flat['tipoResponsabilidad'].astype(str).str.strip().str.upper()
creditos_flat['freq_code'] = creditos_flat['frecuenciaPagos'].astype(str).str.strip().str.upper()
creditos_flat['ta_code'] = creditos_flat['tipoCuenta'].astype(str).str.strip().str.upper()
creditos_flat['tc_code'] = creditos_flat['tipoCredito'].astype(str).str.strip().str.upper()
creditos_flat['no_code'] = creditos_flat['nombreOtorgante'].astype(str).str.strip()

creditos_flat['resp_big'] = creditos_flat['resp_code'].map(resp_to_big).fillna(DEFAULT_OTHER)
creditos_flat['freq_big'] = creditos_flat['freq_code'].map(freq_to_big).fillna(DEFAULT_OTHER)
creditos_flat['ta_big'] = creditos_flat['ta_code'].map(ta_to_big).fillna(DEFAULT_OTHER)
creditos_flat['tc_big'] = creditos_flat['tc_code'].map(tc_to_big).fillna(DEFAULT_OTHER)
creditos_flat['no_big'] = creditos_flat['no_code'].map(no_to_big).fillna(DEFAULT_OTHER)

# 兜底校验输出
print('\n[resp_big] value_counts:')
print(creditos_flat['resp_big'].value_counts(dropna=False).head(20).to_string())

print('\n[freq_big] value_counts:')
print(creditos_flat['freq_big'].value_counts(dropna=False).head(20).to_string())

print('\n[tc_big] value_counts:')
print(creditos_flat['tc_big'].value_counts(dropna=False).head(20).to_string())

print('\n[no_big] value_counts:')
print(creditos_flat['no_big'].value_counts(dropna=False).head(20).to_string())

# 未命中代码样例（便于你检查）
print('\n未命中 resp_code 示例:', creditos_flat.loc[creditos_flat['resp_big']==DEFAULT_OTHER,'resp_code'].dropna().unique()[:20])
print('未命中 freq_code 示例:', creditos_flat.loc[creditos_flat['freq_big']==DEFAULT_OTHER,'freq_code'].dropna().unique()[:20])
print('未命中 ta_code 示例:', creditos_flat.loc[creditos_flat['ta_big']==DEFAULT_OTHER,'ta_code'].dropna().unique()[:20])
print('未命中 tc_code 示例:', creditos_flat.loc[creditos_flat['tc_big']==DEFAULT_OTHER,'tc_code'].dropna().unique()[:20])
print('未命中 no_code 示例:', creditos_flat.loc[creditos_flat['no_big']==DEFAULT_OTHER,'no_code'].dropna().unique()[:20])

creditos_flat.head(10)

cutoff_col = apply_time
creditos_flat.shape = (730191, 24)
event(upd)=fechaActualizacion_dt优先，兜底fechaReporte_dt；缺失率= 0.0
event(open)=fechaAperturaCuenta_dt；缺失率= 0.0
event(lastbuy)=fechaUltimaCompra_dt；缺失率= 0.010760198359059479

[resp_big] value_counts:
resp_big
Individual         694488
SolidaryObligor     17667
HolderAval          14961
Joint                2592
Guarantor             483

[freq_big] value_counts:
freq_big
Monthly         240075
Weekly          217852
BiWeekly        111375
RevolvingMin     76982
OneTime          62430
Other            21477

[tc_big] value_counts:
tc_big
PersonalLoan      445590
CreditLine         77674
ConsumerCredit     73314
CreditCard         37123
ConsumerLoan       19204
PayrollLoan        18782
GroupLoan          17388
Other              14558
Appliance          14388
RealEstate          5545
Guarantee           5430
Auto                 746
Enterprise           449

[no_big] value_counts:
no_big
Retail             149776
PersonalLoanCo     124

,apply_id,cutoff_dt,fechaActualizacion,fechaReporte,fechaAperturaCuenta,fechaUltimaCompra,fechaCierreCuenta,tipoResponsabilidad,frecuenciaPagos,tipoCuenta,...,resp_code,freq_code,ta_code,tc_code,no_code,resp_big,freq_big,ta_big,tc_big,no_big
0,1355047166686003201,2026-03-02 10:44:36.872,2020-06-21,2020-06-21,2019-08-01,2019-08-01,2020-06-21,I,S,F,...,I,S,F,PP,MERCANCIA PARA HOGAR Y OFICINA,Individual,Weekly,FixedPay,PersonalLoan,Retail
1,1355047166686003201,2026-03-02 10:44:36.872,2020-05-29,2020-05-29,2019-08-01,2019-08-01,2020-05-28,I,S,F,...,I,S,F,PP,MERCANCIA PARA HOGAR Y OFICINA,Individual,Weekly,FixedPay,PersonalLoan,Retail
2,1355047166686003201,2026-03-02 10:44:36.872,2026-02-19,2026-02-19,2020-07-26,2020-07-26,NaN,I,S,F,...,I,S,F,PP,MERCANCIA PARA HOGAR Y OFICINA,Individual,Weekly,FixedPay,PersonalLoan,Retail
3,1355047166686003201,2026-03-02 10:44:36.872,2026-02-19,2026-02-19,2020-07-26,2020-07-26,NaN,I,S,F,...,I,S,F,PP,MERCANCIA PARA HOGAR Y OFICINA,Individual,Weekly,FixedPay,PersonalLoan,Retail
4,1355047166686003201,2026-03-02 10:44:36.872,2026-02-19,2026-02-19,2020-10-15,2020-10-15,NaN,I,S,F,...,I,S,F,PP,MERCANCIA PARA HOGAR Y OFICINA,Individual,Weekly,FixedPay,PersonalLoan,Retail
5,1355047166686003201,2026-03-02 10:44:36.872,2026-02-19,2026-02-19,2020-10-15,2020-10-15,NaN,I,S,F,...,I,S,F,PP,MERCANCIA PARA HOGAR Y OFICINA,Individual,Weekly,FixedPay,PersonalLoan,Retail
6,1355047166686003201,2026-03-02 10:44:36.872,2019-04-30,2019-04-30,2018-11-14,2018-11-14,2019-04-03,I,C,F,...,I,C,F,PP,BANCOS,Individual,Other,FixedPay,PersonalLoan,Bank
7,1355047166686003201,2026-03-02 10:44:36.872,2019-08-31,2019-08-31,2019-04-03,2019-04-03,2019-08-21,I,C,F,...,I,C,F,PP,BANCOS,Individual,Other,FixedPay,PersonalLoan,Bank
8,1355047166686003201,2026-03-02 10:44:36.872,2020-01-31,2020-01-31,2019-08-30,2019-08-30,2020-01-16,I,C,F,...,I,C,F,PP,BANCOS,Individual,Other,FixedPay,PersonalLoan,Bank
9,1355047166686003201,2026-03-02 10:44:36.872,2020-08-31,2020-08-31,2020-01-31,2020-01-31,2020-08-13,I,C,F,...,I,C,F,PP,BANCOS,Individual,Other,FixedPay,PersonalLoan,Bank


In [6]:
import re
import numpy as np
import pandas as pd

# =========================
# 信用板块（creditos）衍生：35个指标
# - 窗口：WINDOW_DAYS
# - 截止日：cutoff_dt
# - 事件日：event_dt = fechaActualizacion_dt
# - 分组大类：resp_big / freq_big / tc_big / no_big（均为英文大类名，未命中=Other）
# - 数值字段（空值参与规则）：计算时把空值当0，但不对原字段做填充
# =========================

all_apply_ids = df_raw['apply_id'].astype(str).tolist()

# 数值字段：转换为数值（不填充），后续计算时再按需把 NaN 当 0
NUM_COLS = [
    'saldoVencido',        # 逾期余额
    'limiteCredito',       # 信用额度
    'saldoActual',         # 当前待还
    'montoPagar',          # 当期应还
    'valorActivoValuacion',# 资产估值
    'numeroPagos',         # 付款次数
    'peorAtraso',          # 最严重逾期
]

# 确保这些字段在 creditos_flat 中存在（若缺失则创建空列，避免代码报错；实际输出会全是0/Other）
for c in NUM_COLS:
    if c not in creditos_flat.columns:
        creditos_flat[c] = np.nan

for c in NUM_COLS:
    creditos_flat[c + '_num'] = pd.to_numeric(creditos_flat[c], errors='coerce')

# 参与35个指标计算的“数值字段列”（已转成 *_num），用于判断“该类别下是否全缺失”
NUM_NUM_COLS = [c + '_num' for c in NUM_COLS]


def safe_token(x: str) -> str:
    return re.sub(r'[^A-Za-z0-9_]+', '', str(x))


def pivot_fill(series: pd.Series, index: list[str], columns: list[str], fill_value):
    wide = series.unstack(fill_value=fill_value)
    return wide.reindex(index=index, columns=columns, fill_value=fill_value)


def calc_metrics_by_group(
    df_win: pd.DataFrame,
    group_col: str,
    group_values: list[str],
    w: int,
    prefix: str,
    total_cnt: pd.Series,
    all_cnt: pd.Series,
):
    """对某个分组大类计算指标。

    关键口径：
    - “占比 ratio” = 窗口内该类总笔数 / 全量(36000窗口)总笔数（若分母=0则0）
    - 其它占比（如逾期占比/还清占比等）仍按：该类数量 / 窗口内该类总笔数

    缺失口径补充（你刚刚新增的规则）：
    - 若某客户在该窗口内“确实有该类别记录”，但该类别下参与35个指标的所有数值字段（NUM_NUM_COLS）全部缺失，
      则该客户在该类别下衍生出的所有指标统一置为 -999（避免把“没信息”伪装成0）。
    """

    # 先判断：每个(客户, 类别)是否存在“任一数值字段非空”
    # 注意：这里用 *_num 的 notna，不做 fillna(0)，确保能识别“全缺失”。
    row_has_any_num = df_win[NUM_NUM_COLS].notna().any(axis=1)
    any_num_pair = row_has_any_num.groupby([df_win['apply_id'], df_win[group_col]]).any()
    any_num_wide = any_num_pair.unstack(fill_value=False).reindex(index=all_apply_ids, columns=group_values, fill_value=False)

    # 取数值（计算时把 NaN 当 0）
    saldoV = df_win['saldoVencido_num'].fillna(0)
    limC = df_win['limiteCredito_num'].fillna(0)
    saldoA = df_win['saldoActual_num']  # paid_off 要用 notna
    saldoA0 = saldoA.fillna(0)
    montoP = df_win['montoPagar_num'].fillna(0)
    valAct = df_win['valorActivoValuacion_num'].fillna(0)
    numPag = df_win['numeroPagos_num'].fillna(0)
    peorAt = df_win['peorAtraso_num'].fillna(0)

    gb = df_win.groupby(['apply_id', group_col])

    cnt = gb.size()
    sum_saldoV = saldoV.groupby([df_win['apply_id'], df_win[group_col]]).sum()
    sum_limC = limC.groupby([df_win['apply_id'], df_win[group_col]]).sum()
    sum_saldoA = saldoA0.groupby([df_win['apply_id'], df_win[group_col]]).sum()
    sum_montoP = montoP.groupby([df_win['apply_id'], df_win[group_col]]).sum()
    sum_numPag = numPag.groupby([df_win['apply_id'], df_win[group_col]]).sum()

    max_saldoA = saldoA0.groupby([df_win['apply_id'], df_win[group_col]]).max()
    max_limC = limC.groupby([df_win['apply_id'], df_win[group_col]]).max()

    # 计数类（空值当0）
    limit_pos_cnt = (limC > 0).groupby([df_win['apply_id'], df_win[group_col]]).sum()
    overdue_cnt = (saldoV > 0).groupby([df_win['apply_id'], df_win[group_col]]).sum()

    paid_off_cnt = ((saldoA.notna()) & (saldoA == 0)).groupby([df_win['apply_id'], df_win[group_col]]).sum()
    unpaid_cnt = (montoP > 0).groupby([df_win['apply_id'], df_win[group_col]]).sum()

    asset_gt10k_cnt = (valAct > 10000).groupby([df_win['apply_id'], df_win[group_col]]).sum()
    asset_gt1k_cnt = (valAct > 1000).groupby([df_win['apply_id'], df_win[group_col]]).sum()

    numPag_ge5_cnt = (numPag >= 5).groupby([df_win['apply_id'], df_win[group_col]]).sum()
    numPag_eq1_cnt = (numPag == 1).groupby([df_win['apply_id'], df_win[group_col]]).sum()
    numPag_1to5_cnt = ((numPag > 1) & (numPag < 5)).groupby([df_win['apply_id'], df_win[group_col]]).sum()

    peor_ge10_cnt = (peorAt >= 10).groupby([df_win['apply_id'], df_win[group_col]]).sum()
    peor_ge1_cnt = (peorAt >= 1).groupby([df_win['apply_id'], df_win[group_col]]).sum()
    peor_ge2_cnt = (peorAt >= 2).groupby([df_win['apply_id'], df_win[group_col]]).sum()

    # wide
    w_cnt = pivot_fill(cnt, all_apply_ids, group_values, 0).astype('int32')
    w_sum_saldoV = pivot_fill(sum_saldoV, all_apply_ids, group_values, 0.0)
    w_sum_limC = pivot_fill(sum_limC, all_apply_ids, group_values, 0.0)
    w_sum_saldoA = pivot_fill(sum_saldoA, all_apply_ids, group_values, 0.0)
    w_sum_montoP = pivot_fill(sum_montoP, all_apply_ids, group_values, 0.0)
    w_max_saldoA = pivot_fill(max_saldoA, all_apply_ids, group_values, 0.0)
    w_max_limC = pivot_fill(max_limC, all_apply_ids, group_values, 0.0)

    w_limit_pos_cnt = pivot_fill(limit_pos_cnt, all_apply_ids, group_values, 0).astype('int32')
    w_overdue_cnt = pivot_fill(overdue_cnt, all_apply_ids, group_values, 0).astype('int32')
    w_paid_off_cnt = pivot_fill(paid_off_cnt, all_apply_ids, group_values, 0).astype('int32')
    w_unpaid_cnt = pivot_fill(unpaid_cnt, all_apply_ids, group_values, 0).astype('int32')

    w_asset_gt10k_cnt = pivot_fill(asset_gt10k_cnt, all_apply_ids, group_values, 0).astype('int32')
    w_asset_gt1k_cnt = pivot_fill(asset_gt1k_cnt, all_apply_ids, group_values, 0).astype('int32')

    w_sum_numPag = pivot_fill(sum_numPag, all_apply_ids, group_values, 0.0)
    w_numPag_ge5_cnt = pivot_fill(numPag_ge5_cnt, all_apply_ids, group_values, 0).astype('int32')
    w_numPag_eq1_cnt = pivot_fill(numPag_eq1_cnt, all_apply_ids, group_values, 0).astype('int32')
    w_numPag_1to5_cnt = pivot_fill(numPag_1to5_cnt, all_apply_ids, group_values, 0).astype('int32')

    w_peor_ge10_cnt = pivot_fill(peor_ge10_cnt, all_apply_ids, group_values, 0).astype('int32')
    w_peor_ge1_cnt = pivot_fill(peor_ge1_cnt, all_apply_ids, group_values, 0).astype('int32')
    w_peor_ge2_cnt = pivot_fill(peor_ge2_cnt, all_apply_ids, group_values, 0).astype('int32')

    # denom
    w_cnt_f = w_cnt.astype('float64')
    all_cnt_f = all_cnt.astype('float64').replace(0, np.nan)

    cols = {}
    for cls in group_values:
        t = safe_token(cls)

        cnt_cls = w_cnt[cls]
        cnt_cls_f = w_cnt_f[cls]

        # 该客户在该类别下：有记录但所有数值字段都缺失 -> 该类别所有指标置为 -999
        bad_cls = (cnt_cls > 0) & (~any_num_wide[cls])

        sum_lim_f = w_sum_limC[cls].astype('float64')

        # 1 金额逾期率：sum(saldoVencido)/sum(limiteCredito)
        denom_lim = sum_lim_f.replace(0, np.nan)
        overdue_rate = (w_sum_saldoV[cls].astype('float64') / denom_lim).fillna(0.0)
        overdue_rate = overdue_rate.mask(bad_cls, -999.0)
        cols[f'{prefix}_{t}_{w}d_overdue_rate'] = overdue_rate.astype('float32').to_numpy()

        # 2 平均信用额度：sum(limiteCredito)/cnt
        denom_cnt = cnt_cls_f.replace(0, np.nan)
        avg_limit = (sum_lim_f / denom_cnt).fillna(0.0)
        avg_limit = avg_limit.mask(bad_cls, -999.0)
        cols[f'{prefix}_{t}_{w}d_avg_limit'] = avg_limit.astype('float32').to_numpy()

        # 3 数量
        cnt_out = cnt_cls.astype('int32').mask(bad_cls, -999)
        cols[f'{prefix}_{t}_{w}d_cnt'] = cnt_out.astype('int32').to_numpy()

        # 4 有效额度数量（limiteCredito>0）
        limit_pos_out = w_limit_pos_cnt[cls].mask(bad_cls, -999).astype('int32')
        cols[f'{prefix}_{t}_{w}d_limit_pos_cnt'] = limit_pos_out.to_numpy()

        # 5 逾期数量（saldoVencido>0）
        overdue_cnt_out = w_overdue_cnt[cls].mask(bad_cls, -999).astype('int32')
        cols[f'{prefix}_{t}_{w}d_overdue_cnt'] = overdue_cnt_out.to_numpy()

        # 6 逾期占比（逾期数量/该类总笔数）
        overdue_ratio = (w_overdue_cnt[cls].astype('float64') / denom_cnt).fillna(0.0)
        overdue_ratio = overdue_ratio.mask(bad_cls, -999.0)
        cols[f'{prefix}_{t}_{w}d_overdue_ratio'] = overdue_ratio.astype('float32').to_numpy()

        # 7 占比（该类笔数/全量总笔数）
        ratio = (cnt_cls_f / all_cnt_f).fillna(0.0)
        ratio = ratio.mask(bad_cls, -999.0)
        cols[f'{prefix}_{t}_{w}d_ratio'] = ratio.astype('float32').to_numpy()

        # 8 总剩余待还金额
        cols[f'{prefix}_{t}_{w}d_sum_saldoActual'] = w_sum_saldoA[cls].astype('float32').mask(bad_cls, -999.0).to_numpy()

        # 9 总信用额度
        cols[f'{prefix}_{t}_{w}d_sum_limiteCredito'] = w_sum_limC[cls].astype('float32').mask(bad_cls, -999.0).to_numpy()

        # 10 总逾期剩余待还金额
        cols[f'{prefix}_{t}_{w}d_sum_saldoVencido'] = w_sum_saldoV[cls].astype('float32').mask(bad_cls, -999.0).to_numpy()

        # 11 最大剩余待还金额
        cols[f'{prefix}_{t}_{w}d_max_saldoActual'] = w_max_saldoA[cls].astype('float32').mask(bad_cls, -999.0).to_numpy()

        # 12 最大信用额度
        cols[f'{prefix}_{t}_{w}d_max_limiteCredito'] = w_max_limC[cls].astype('float32').mask(bad_cls, -999.0).to_numpy()

        # 13 最近应还余额不含坏账（montoPagar求和）
        cols[f'{prefix}_{t}_{w}d_sum_montoPagar'] = w_sum_montoP[cls].astype('float32').mask(bad_cls, -999.0).to_numpy()

        # 14 还清借款数量（saldoActual不为空且==0）
        cols[f'{prefix}_{t}_{w}d_paid_off_cnt'] = w_paid_off_cnt[cls].mask(bad_cls, -999).astype('int32').to_numpy()

        # 15 还清借款占比
        paid_off_ratio = (w_paid_off_cnt[cls].astype('float64') / denom_cnt).fillna(0.0)
        paid_off_ratio = paid_off_ratio.mask(bad_cls, -999.0)
        cols[f'{prefix}_{t}_{w}d_paid_off_ratio'] = paid_off_ratio.astype('float32').to_numpy()

        # 16 未还清借款数量（montoPagar>0）
        cols[f'{prefix}_{t}_{w}d_unpaid_cnt'] = w_unpaid_cnt[cls].mask(bad_cls, -999).astype('int32').to_numpy()

        # 17 未还清借款占比
        unpaid_ratio = (w_unpaid_cnt[cls].astype('float64') / denom_cnt).fillna(0.0)
        unpaid_ratio = unpaid_ratio.mask(bad_cls, -999.0)
        cols[f'{prefix}_{t}_{w}d_unpaid_ratio'] = unpaid_ratio.astype('float32').to_numpy()

        # 18 资产价值>10k 数量
        cols[f'{prefix}_{t}_{w}d_asset_gt10k_cnt'] = w_asset_gt10k_cnt[cls].mask(bad_cls, -999).astype('int32').to_numpy()

        # 19 资产价值>10k 占比
        asset_gt10k_ratio = (w_asset_gt10k_cnt[cls].astype('float64') / denom_cnt).fillna(0.0)
        asset_gt10k_ratio = asset_gt10k_ratio.mask(bad_cls, -999.0)
        cols[f'{prefix}_{t}_{w}d_asset_gt10k_ratio'] = asset_gt10k_ratio.astype('float32').to_numpy()

        # 20 资产价值>1k 数量
        cols[f'{prefix}_{t}_{w}d_asset_gt1k_cnt'] = w_asset_gt1k_cnt[cls].mask(bad_cls, -999).astype('int32').to_numpy()

        # 21 资产价值>1k 占比
        asset_gt1k_ratio = (w_asset_gt1k_cnt[cls].astype('float64') / denom_cnt).fillna(0.0)
        asset_gt1k_ratio = asset_gt1k_ratio.mask(bad_cls, -999.0)
        cols[f'{prefix}_{t}_{w}d_asset_gt1k_ratio'] = asset_gt1k_ratio.astype('float32').to_numpy()

        # 22 总还款次数（numeroPagos求和）
        cols[f'{prefix}_{t}_{w}d_sum_numeroPagos'] = w_sum_numPag[cls].astype('float32').mask(bad_cls, -999.0).to_numpy()

        # 23 总还款次数>=5 数量
        cols[f'{prefix}_{t}_{w}d_numPag_ge5_cnt'] = w_numPag_ge5_cnt[cls].mask(bad_cls, -999).astype('int32').to_numpy()

        # 24 总还款次数>=5 占比
        numPag_ge5_ratio = (w_numPag_ge5_cnt[cls].astype('float64') / denom_cnt).fillna(0.0)
        numPag_ge5_ratio = numPag_ge5_ratio.mask(bad_cls, -999.0)
        cols[f'{prefix}_{t}_{w}d_numPag_ge5_ratio'] = numPag_ge5_ratio.astype('float32').to_numpy()

        # 25 总还款次数==1 数量
        cols[f'{prefix}_{t}_{w}d_numPag_eq1_cnt'] = w_numPag_eq1_cnt[cls].mask(bad_cls, -999).astype('int32').to_numpy()

        # 26 总还款次数==1 占比
        numPag_eq1_ratio = (w_numPag_eq1_cnt[cls].astype('float64') / denom_cnt).fillna(0.0)
        numPag_eq1_ratio = numPag_eq1_ratio.mask(bad_cls, -999.0)
        cols[f'{prefix}_{t}_{w}d_numPag_eq1_ratio'] = numPag_eq1_ratio.astype('float32').to_numpy()

        # 27 总还款次数(1,5) 数量
        cols[f'{prefix}_{t}_{w}d_numPag_1to5_cnt'] = w_numPag_1to5_cnt[cls].mask(bad_cls, -999).astype('int32').to_numpy()

        # 28 总还款次数(1,5) 占比
        numPag_1to5_ratio = (w_numPag_1to5_cnt[cls].astype('float64') / denom_cnt).fillna(0.0)
        numPag_1to5_ratio = numPag_1to5_ratio.mask(bad_cls, -999.0)
        cols[f'{prefix}_{t}_{w}d_numPag_1to5_ratio'] = numPag_1to5_ratio.astype('float32').to_numpy()

        # 29 最大逾期>=10 数量
        cols[f'{prefix}_{t}_{w}d_peor_ge10_cnt'] = w_peor_ge10_cnt[cls].mask(bad_cls, -999).astype('int32').to_numpy()

        # 30 最大逾期>=10 占比
        peor_ge10_ratio = (w_peor_ge10_cnt[cls].astype('float64') / denom_cnt).fillna(0.0)
        peor_ge10_ratio = peor_ge10_ratio.mask(bad_cls, -999.0)
        cols[f'{prefix}_{t}_{w}d_peor_ge10_ratio'] = peor_ge10_ratio.astype('float32').to_numpy()

        # 31 最大逾期>=1 数量
        cols[f'{prefix}_{t}_{w}d_peor_ge1_cnt'] = w_peor_ge1_cnt[cls].mask(bad_cls, -999).astype('int32').to_numpy()

        # 32 最大逾期>=1 占比
        peor_ge1_ratio = (w_peor_ge1_cnt[cls].astype('float64') / denom_cnt).fillna(0.0)
        peor_ge1_ratio = peor_ge1_ratio.mask(bad_cls, -999.0)
        cols[f'{prefix}_{t}_{w}d_peor_ge1_ratio'] = peor_ge1_ratio.astype('float32').to_numpy()

        # 33 最大逾期>=2 数量
        cols[f'{prefix}_{t}_{w}d_peor_ge2_cnt'] = w_peor_ge2_cnt[cls].mask(bad_cls, -999).astype('int32').to_numpy()

        # 34 最大逾期>=2 占比
        peor_ge2_ratio = (w_peor_ge2_cnt[cls].astype('float64') / denom_cnt).fillna(0.0)
        peor_ge2_ratio = peor_ge2_ratio.mask(bad_cls, -999.0)
        cols[f'{prefix}_{t}_{w}d_peor_ge2_ratio'] = peor_ge2_ratio.astype('float32').to_numpy()

    return cols


# 分组大类列表（保证每个窗口列一致）
resp_classes = sorted(set(resp_to_big.values()) | {DEFAULT_OTHER})
freq_classes = sorted(set(freq_to_big.values()) | {DEFAULT_OTHER})
tc_classes = sorted(set(tc_to_big.values()) | {DEFAULT_OTHER})
no_classes = sorted(set(no_to_big.values()) | {DEFAULT_OTHER})

# 3 套窗口计算日期：upd/open/lastbuy
DATE_SPECS = [
    ('upd', 'diff_days_upd'),
    ('open', 'diff_days_open'),
    ('lastbuy', 'diff_days_lastbuy'),
]


def calc_metrics_all(df_win: pd.DataFrame, tag: str, w: int, total_cnt: pd.Series, all_cnt: pd.Series):
    """不分大类：每客户×每窗口，计算同款指标（共35项，含借款总数）。

    缺失口径补充：
    - 若某客户在该窗口内有 creditos 记录（total_cnt>0），但参与35个指标的所有数值字段（NUM_NUM_COLS）全部缺失，
      则该客户在该窗口下“不分类型”的35个指标统一置为 -999。
    """

    # 有记录但所有数值字段都缺失 -> 该窗口 all 指标置为 -999
    row_has_any_num = df_win[NUM_NUM_COLS].notna().any(axis=1)
    has_any_num = row_has_any_num.groupby(df_win['apply_id']).any().reindex(all_apply_ids, fill_value=False)
    bad_all = (total_cnt > 0) & (~has_any_num)

    saldoV = df_win['saldoVencido_num'].fillna(0)
    limC = df_win['limiteCredito_num'].fillna(0)
    saldoA = df_win['saldoActual_num']
    saldoA0 = saldoA.fillna(0)
    montoP = df_win['montoPagar_num'].fillna(0)
    valAct = df_win['valorActivoValuacion_num'].fillna(0)
    numPag = df_win['numeroPagos_num'].fillna(0)
    peorAt = df_win['peorAtraso_num'].fillna(0)

    g = df_win.groupby('apply_id')

    sum_saldoV = saldoV.groupby(df_win['apply_id']).sum().reindex(all_apply_ids, fill_value=0.0)
    sum_limC = limC.groupby(df_win['apply_id']).sum().reindex(all_apply_ids, fill_value=0.0)
    sum_saldoA = saldoA0.groupby(df_win['apply_id']).sum().reindex(all_apply_ids, fill_value=0.0)
    sum_montoP = montoP.groupby(df_win['apply_id']).sum().reindex(all_apply_ids, fill_value=0.0)

    max_saldoA = saldoA0.groupby(df_win['apply_id']).max().reindex(all_apply_ids, fill_value=0.0)
    max_limC = limC.groupby(df_win['apply_id']).max().reindex(all_apply_ids, fill_value=0.0)

    limit_pos_cnt = (limC > 0).groupby(df_win['apply_id']).sum().reindex(all_apply_ids, fill_value=0).astype('int32')
    overdue_cnt = (saldoV > 0).groupby(df_win['apply_id']).sum().reindex(all_apply_ids, fill_value=0).astype('int32')

    paid_off_cnt = ((saldoA.notna()) & (saldoA == 0)).groupby(df_win['apply_id']).sum().reindex(all_apply_ids, fill_value=0).astype('int32')
    unpaid_cnt = (montoP > 0).groupby(df_win['apply_id']).sum().reindex(all_apply_ids, fill_value=0).astype('int32')

    asset_gt10k_cnt = (valAct > 10000).groupby(df_win['apply_id']).sum().reindex(all_apply_ids, fill_value=0).astype('int32')
    asset_gt1k_cnt = (valAct > 1000).groupby(df_win['apply_id']).sum().reindex(all_apply_ids, fill_value=0).astype('int32')

    sum_numPag = numPag.groupby(df_win['apply_id']).sum().reindex(all_apply_ids, fill_value=0.0)
    numPag_ge5_cnt = (numPag >= 5).groupby(df_win['apply_id']).sum().reindex(all_apply_ids, fill_value=0).astype('int32')
    numPag_eq1_cnt = (numPag == 1).groupby(df_win['apply_id']).sum().reindex(all_apply_ids, fill_value=0).astype('int32')
    numPag_1to5_cnt = ((numPag > 1) & (numPag < 5)).groupby(df_win['apply_id']).sum().reindex(all_apply_ids, fill_value=0).astype('int32')

    peor_ge10_cnt = (peorAt >= 10).groupby(df_win['apply_id']).sum().reindex(all_apply_ids, fill_value=0).astype('int32')
    peor_ge1_cnt = (peorAt >= 1).groupby(df_win['apply_id']).sum().reindex(all_apply_ids, fill_value=0).astype('int32')
    peor_ge2_cnt = (peorAt >= 2).groupby(df_win['apply_id']).sum().reindex(all_apply_ids, fill_value=0).astype('int32')

    total_cnt_f = total_cnt.astype('float64').replace(0, np.nan)
    all_cnt_f = all_cnt.astype('float64').replace(0, np.nan)

    # 1 金额逾期率
    overdue_rate = (sum_saldoV.astype('float64') / sum_limC.astype('float64').replace(0, np.nan)).fillna(0.0)
    # 2 平均信用额度
    avg_limit = (sum_limC.astype('float64') / total_cnt_f).fillna(0.0)
    # 6/15/17/19/21/24/26/28/30/32/34 等占比
    overdue_ratio = (overdue_cnt.astype('float64') / total_cnt_f).fillna(0.0)
    ratio = (total_cnt.astype('float64') / all_cnt_f).fillna(0.0)
    paid_off_ratio = (paid_off_cnt.astype('float64') / total_cnt_f).fillna(0.0)
    unpaid_ratio = (unpaid_cnt.astype('float64') / total_cnt_f).fillna(0.0)
    asset_gt10k_ratio = (asset_gt10k_cnt.astype('float64') / total_cnt_f).fillna(0.0)
    asset_gt1k_ratio = (asset_gt1k_cnt.astype('float64') / total_cnt_f).fillna(0.0)
    numPag_ge5_ratio = (numPag_ge5_cnt.astype('float64') / total_cnt_f).fillna(0.0)
    numPag_eq1_ratio = (numPag_eq1_cnt.astype('float64') / total_cnt_f).fillna(0.0)
    numPag_1to5_ratio = (numPag_1to5_cnt.astype('float64') / total_cnt_f).fillna(0.0)
    peor_ge10_ratio = (peor_ge10_cnt.astype('float64') / total_cnt_f).fillna(0.0)
    peor_ge1_ratio = (peor_ge1_cnt.astype('float64') / total_cnt_f).fillna(0.0)
    peor_ge2_ratio = (peor_ge2_cnt.astype('float64') / total_cnt_f).fillna(0.0)

    # 对 bad_all 的客户，把 all 口径下的指标统一置为 -999
    overdue_rate = overdue_rate.mask(bad_all, -999.0)
    avg_limit = avg_limit.mask(bad_all, -999.0)
    total_cnt_out = total_cnt.astype('int32').mask(bad_all, -999)
    limit_pos_out = limit_pos_cnt.mask(bad_all, -999).astype('int32')
    overdue_cnt_out = overdue_cnt.mask(bad_all, -999).astype('int32')
    overdue_ratio = overdue_ratio.mask(bad_all, -999.0)
    ratio = ratio.mask(bad_all, -999.0)
    sum_saldoA = sum_saldoA.mask(bad_all, -999.0)
    sum_limC = sum_limC.mask(bad_all, -999.0)
    sum_saldoV = sum_saldoV.mask(bad_all, -999.0)
    max_saldoA = max_saldoA.mask(bad_all, -999.0)
    max_limC = max_limC.mask(bad_all, -999.0)
    sum_montoP = sum_montoP.mask(bad_all, -999.0)
    paid_off_cnt_out = paid_off_cnt.mask(bad_all, -999).astype('int32')
    paid_off_ratio = paid_off_ratio.mask(bad_all, -999.0)
    all_cnt_out = all_cnt.astype('int32').mask(bad_all, -999)
    unpaid_cnt_out = unpaid_cnt.mask(bad_all, -999).astype('int32')
    unpaid_ratio = unpaid_ratio.mask(bad_all, -999.0)
    asset_gt10k_cnt_out = asset_gt10k_cnt.mask(bad_all, -999).astype('int32')
    asset_gt10k_ratio = asset_gt10k_ratio.mask(bad_all, -999.0)
    asset_gt1k_cnt_out = asset_gt1k_cnt.mask(bad_all, -999).astype('int32')
    asset_gt1k_ratio = asset_gt1k_ratio.mask(bad_all, -999.0)
    sum_numPag = sum_numPag.mask(bad_all, -999.0)
    numPag_ge5_cnt_out = numPag_ge5_cnt.mask(bad_all, -999).astype('int32')
    numPag_ge5_ratio = numPag_ge5_ratio.mask(bad_all, -999.0)
    numPag_eq1_cnt_out = numPag_eq1_cnt.mask(bad_all, -999).astype('int32')
    numPag_eq1_ratio = numPag_eq1_ratio.mask(bad_all, -999.0)
    numPag_1to5_cnt_out = numPag_1to5_cnt.mask(bad_all, -999).astype('int32')
    numPag_1to5_ratio = numPag_1to5_ratio.mask(bad_all, -999.0)
    peor_ge10_cnt_out = peor_ge10_cnt.mask(bad_all, -999).astype('int32')
    peor_ge10_ratio = peor_ge10_ratio.mask(bad_all, -999.0)
    peor_ge1_cnt_out = peor_ge1_cnt.mask(bad_all, -999).astype('int32')
    peor_ge1_ratio = peor_ge1_ratio.mask(bad_all, -999.0)
    peor_ge2_cnt_out = peor_ge2_cnt.mask(bad_all, -999).astype('int32')
    peor_ge2_ratio = peor_ge2_ratio.mask(bad_all, -999.0)

    cols = {
        f'cr_{tag}_all_{w}d_overdue_rate': overdue_rate.astype('float32').to_numpy(),
        f'cr_{tag}_all_{w}d_avg_limit': avg_limit.astype('float32').to_numpy(),
        f'cr_{tag}_all_{w}d_cnt': total_cnt_out.astype('int32').to_numpy(),
        f'cr_{tag}_all_{w}d_limit_pos_cnt': limit_pos_out.to_numpy(),
        f'cr_{tag}_all_{w}d_overdue_cnt': overdue_cnt_out.to_numpy(),
        f'cr_{tag}_all_{w}d_overdue_ratio': overdue_ratio.astype('float32').to_numpy(),
        f'cr_{tag}_all_{w}d_ratio': ratio.astype('float32').to_numpy(),
        f'cr_{tag}_all_{w}d_sum_saldoActual': sum_saldoA.astype('float32').to_numpy(),
        f'cr_{tag}_all_{w}d_sum_limiteCredito': sum_limC.astype('float32').to_numpy(),
        f'cr_{tag}_all_{w}d_sum_saldoVencido': sum_saldoV.astype('float32').to_numpy(),
        f'cr_{tag}_all_{w}d_max_saldoActual': max_saldoA.astype('float32').to_numpy(),
        f'cr_{tag}_all_{w}d_max_limiteCredito': max_limC.astype('float32').to_numpy(),
        f'cr_{tag}_all_{w}d_sum_montoPagar': sum_montoP.astype('float32').to_numpy(),
        f'cr_{tag}_all_{w}d_paid_off_cnt': paid_off_cnt_out.to_numpy(),
        f'cr_{tag}_all_{w}d_paid_off_ratio': paid_off_ratio.astype('float32').to_numpy(),
        f'cr_{tag}_all_{w}d_borrow_total_cnt': all_cnt_out.astype('int32').to_numpy(),
        f'cr_{tag}_all_{w}d_unpaid_cnt': unpaid_cnt_out.to_numpy(),
        f'cr_{tag}_all_{w}d_unpaid_ratio': unpaid_ratio.astype('float32').to_numpy(),
        f'cr_{tag}_all_{w}d_asset_gt10k_cnt': asset_gt10k_cnt_out.to_numpy(),
        f'cr_{tag}_all_{w}d_asset_gt10k_ratio': asset_gt10k_ratio.astype('float32').to_numpy(),
        f'cr_{tag}_all_{w}d_asset_gt1k_cnt': asset_gt1k_cnt_out.to_numpy(),
        f'cr_{tag}_all_{w}d_asset_gt1k_ratio': asset_gt1k_ratio.astype('float32').to_numpy(),
        f'cr_{tag}_all_{w}d_sum_numeroPagos': sum_numPag.astype('float32').to_numpy(),
        f'cr_{tag}_all_{w}d_numPag_ge5_cnt': numPag_ge5_cnt_out.to_numpy(),
        f'cr_{tag}_all_{w}d_numPag_ge5_ratio': numPag_ge5_ratio.astype('float32').to_numpy(),
        f'cr_{tag}_all_{w}d_numPag_eq1_cnt': numPag_eq1_cnt_out.to_numpy(),
        f'cr_{tag}_all_{w}d_numPag_eq1_ratio': numPag_eq1_ratio.astype('float32').to_numpy(),
        f'cr_{tag}_all_{w}d_numPag_1to5_cnt': numPag_1to5_cnt_out.to_numpy(),
        f'cr_{tag}_all_{w}d_numPag_1to5_ratio': numPag_1to5_ratio.astype('float32').to_numpy(),
        f'cr_{tag}_all_{w}d_peor_ge10_cnt': peor_ge10_cnt_out.to_numpy(),
        f'cr_{tag}_all_{w}d_peor_ge10_ratio': peor_ge10_ratio.astype('float32').to_numpy(),
        f'cr_{tag}_all_{w}d_peor_ge1_cnt': peor_ge1_cnt_out.to_numpy(),
        f'cr_{tag}_all_{w}d_peor_ge1_ratio': peor_ge1_ratio.astype('float32').to_numpy(),
        f'cr_{tag}_all_{w}d_peor_ge2_cnt': peor_ge2_cnt_out.to_numpy(),
        f'cr_{tag}_all_{w}d_peor_ge2_ratio': peor_ge2_ratio.astype('float32').to_numpy(),
    }
    return cols


feat_parts = []

for tag, diff_col in DATE_SPECS:
    # 全量总笔数（用于“占比 ratio”的分母）：窗口内该类总笔数 / 全量总笔数
    m_all = creditos_flat[diff_col].between(0, 36000, inclusive='both')
    all_cnt = creditos_flat.loc[m_all].groupby('apply_id').size().reindex(all_apply_ids, fill_value=0).astype('int32')

    for w in WINDOW_DAYS:
        m = creditos_flat[diff_col].between(0, w, inclusive='both')
        df_win = creditos_flat.loc[m].copy()

        # 窗口总笔数（所有类）
        total_cnt = df_win.groupby('apply_id').size().reindex(all_apply_ids, fill_value=0).astype('int32')

        cols = {}
        cols[f'cr_{tag}_win_{w}d_total_cnt'] = total_cnt.to_numpy()

        # 不分大类：每客户×每窗口（35项）
        cols.update(calc_metrics_all(df_win, tag=tag, w=w, total_cnt=total_cnt, all_cnt=all_cnt))

        # 四套大类分别计算（ratio口径：该类/全量总笔数）
        cols.update(calc_metrics_by_group(df_win, 'resp_big', resp_classes, w, prefix=f'cr_{tag}_resp', total_cnt=total_cnt, all_cnt=all_cnt))
        cols.update(calc_metrics_by_group(df_win, 'freq_big', freq_classes, w, prefix=f'cr_{tag}_freq', total_cnt=total_cnt, all_cnt=all_cnt))
        cols.update(calc_metrics_by_group(df_win, 'tc_big', tc_classes, w, prefix=f'cr_{tag}_tc', total_cnt=total_cnt, all_cnt=all_cnt))
        cols.update(calc_metrics_by_group(df_win, 'no_big', no_classes, w, prefix=f'cr_{tag}_no', total_cnt=total_cnt, all_cnt=all_cnt))

        part = pd.DataFrame(cols, index=all_apply_ids)
        feat_parts.append(part)
        print('date', tag, 'window', w, 'done; part.shape=', part.shape)

creditos_features = pd.concat(feat_parts, axis=1)
creditos_features.insert(0, 'apply_id', all_apply_ids)

# 添加cust_no列（从df_raw中获取）
cust_no_map = df_raw.set_index('apply_id')['cust_no']
creditos_features.insert(1, 'cust_no', creditos_features['apply_id'].map(cust_no_map))

# 添加create_time列（从df_raw中获取apply_time）
create_time_map = df_raw.set_index('apply_id')['apply_time']
creditos_features.insert(2, 'create_time', creditos_features['apply_id'].map(create_time_map))

# =========================
# 对“信用板块整行缺失”的客户：整行特征置为 -999
# 口径：
# 1) creditos 完全缺失（该客户一条 creditos 记录都没有） -> 全部特征 -999
# 2) 该客户 creditos 记录存在，但三套事件日期（upd/open/lastbuy）全部缺失 -> 全部特征 -999
# 说明：
# - 若只是“没有某个类别/某窗口内没数据”，数量/求和/计数/均值/比例都按 0 输出（表示没发生），不视为缺失。
# - 数值字段若部分缺失，计算时按 0 参与聚合；仅在“整行缺失”时才整体置 -999。
# =========================

# 该客户是否存在任何 creditos 记录
has_any_creditos = creditos_flat.groupby('apply_id').size().reindex(all_apply_ids, fill_value=0) > 0

# 该客户是否存在任何事件日期（任一口径 event_dt 非空）
EVENT_COLS = ['event_dt_upd', 'event_dt_open', 'event_dt_lastbuy']
for c in EVENT_COLS:
    if c not in creditos_flat.columns:
        creditos_flat[c] = pd.NaT

row_has_event = creditos_flat[EVENT_COLS].notna().any(axis=1)
has_any_event = row_has_event.groupby(creditos_flat['apply_id']).any().reindex(all_apply_ids, fill_value=False)

# 整行缺失：creditos 不存在 或（存在但事件日期全缺失）
bad_mask = (~has_any_creditos) | (has_any_creditos & (~has_any_event))
print('整行缺失客户数(信用板块特征将置为-999):', int(bad_mask.sum()), '/', len(bad_mask))

# 在 creditos_features 里把这些客户的所有特征列置为 -999（保留 apply_id）
feat_cols_all = [c for c in creditos_features.columns if c not in ['apply_id', 'cust_no', 'create_time']]
creditos_features.loc[bad_mask.values, feat_cols_all] = -999

print('creditos_features.shape =', creditos_features.shape)
creditos_features.head(3)

/var/folders/nx/n0480dd568gdv3z1tbz9smp00000gn/T/ipykernel_60736/3169537871.py:44: Pandas4Warning: Using a fill_value that cannot be held in the existing dtype is deprecated and will raise in a future version.
  wide = series.unstack(fill_value=fill_value)
/var/folders/nx/n0480dd568gdv3z1tbz9smp00000gn/T/ipykernel_60736/3169537871.py:44: Pandas4Warning: Using a fill_value that cannot be held in the existing dtype is deprecated and will raise in a future version.
  wide = series.unstack(fill_value=fill_value)
/var/folders/nx/n0480dd568gdv3z1tbz9smp00000gn/T/ipykernel_60736/3169537871.py:44: Pandas4Warning: Using a fill_value that cannot be held in the existing dtype is deprecated and will raise in a future version.
  wide = series.unstack(fill_value=fill_value)
/var/folders/nx/n0480dd568gdv3z1tbz9smp00000gn/T/ipykernel_60736/3169537871.py:44: Pandas4Warning: Using a fill_value that cannot be held in the existing dtype is deprecated and will raise in a future version.
  wide = series.uns

date upd window 30 done; part.shape= (14963, 1498)


/var/folders/nx/n0480dd568gdv3z1tbz9smp00000gn/T/ipykernel_60736/3169537871.py:44: Pandas4Warning: Using a fill_value that cannot be held in the existing dtype is deprecated and will raise in a future version.
  wide = series.unstack(fill_value=fill_value)
/var/folders/nx/n0480dd568gdv3z1tbz9smp00000gn/T/ipykernel_60736/3169537871.py:44: Pandas4Warning: Using a fill_value that cannot be held in the existing dtype is deprecated and will raise in a future version.
  wide = series.unstack(fill_value=fill_value)
/var/folders/nx/n0480dd568gdv3z1tbz9smp00000gn/T/ipykernel_60736/3169537871.py:44: Pandas4Warning: Using a fill_value that cannot be held in the existing dtype is deprecated and will raise in a future version.
  wide = series.unstack(fill_value=fill_value)
/var/folders/nx/n0480dd568gdv3z1tbz9smp00000gn/T/ipykernel_60736/3169537871.py:44: Pandas4Warning: Using a fill_value that cannot be held in the existing dtype is deprecated and will raise in a future version.
  wide = series.uns

date upd window 60 done; part.shape= (14963, 1498)


/var/folders/nx/n0480dd568gdv3z1tbz9smp00000gn/T/ipykernel_60736/3169537871.py:44: Pandas4Warning: Using a fill_value that cannot be held in the existing dtype is deprecated and will raise in a future version.
  wide = series.unstack(fill_value=fill_value)
/var/folders/nx/n0480dd568gdv3z1tbz9smp00000gn/T/ipykernel_60736/3169537871.py:44: Pandas4Warning: Using a fill_value that cannot be held in the existing dtype is deprecated and will raise in a future version.
  wide = series.unstack(fill_value=fill_value)
/var/folders/nx/n0480dd568gdv3z1tbz9smp00000gn/T/ipykernel_60736/3169537871.py:44: Pandas4Warning: Using a fill_value that cannot be held in the existing dtype is deprecated and will raise in a future version.
  wide = series.unstack(fill_value=fill_value)
/var/folders/nx/n0480dd568gdv3z1tbz9smp00000gn/T/ipykernel_60736/3169537871.py:44: Pandas4Warning: Using a fill_value that cannot be held in the existing dtype is deprecated and will raise in a future version.
  wide = series.uns

...

/var/folders/nx/n0480dd568gdv3z1tbz9smp00000gn/T/ipykernel_60736/3169537871.py:44: Pandas4Warning: Using a fill_value that cannot be held in the existing dtype is deprecated and will raise in a future version.
  wide = series.unstack(fill_value=fill_value)
/var/folders/nx/n0480dd568gdv3z1tbz9smp00000gn/T/ipykernel_60736/3169537871.py:44: Pandas4Warning: Using a fill_value that cannot be held in the existing dtype is deprecated and will raise in a future version.
  wide = series.unstack(fill_value=fill_value)
/var/folders/nx/n0480dd568gdv3z1tbz9smp00000gn/T/ipykernel_60736/3169537871.py:44: Pandas4Warning: Using a fill_value that cannot be held in the existing dtype is deprecated and will raise in a future version.
  wide = series.unstack(fill_value=fill_value)
/var/folders/nx/n0480dd568gdv3z1tbz9smp00000gn/T/ipykernel_60736/3169537871.py:44: Pandas4Warning: Using a fill_value that cannot be held in the existing dtype is deprecated and will raise in a future version.
  wide = series.uns

date lastbuy window 720 done; part.shape= (14963, 1498)


/var/folders/nx/n0480dd568gdv3z1tbz9smp00000gn/T/ipykernel_60736/3169537871.py:44: Pandas4Warning: Using a fill_value that cannot be held in the existing dtype is deprecated and will raise in a future version.
  wide = series.unstack(fill_value=fill_value)
/var/folders/nx/n0480dd568gdv3z1tbz9smp00000gn/T/ipykernel_60736/3169537871.py:44: Pandas4Warning: Using a fill_value that cannot be held in the existing dtype is deprecated and will raise in a future version.
  wide = series.unstack(fill_value=fill_value)
/var/folders/nx/n0480dd568gdv3z1tbz9smp00000gn/T/ipykernel_60736/3169537871.py:44: Pandas4Warning: Using a fill_value that cannot be held in the existing dtype is deprecated and will raise in a future version.
  wide = series.unstack(fill_value=fill_value)
/var/folders/nx/n0480dd568gdv3z1tbz9smp00000gn/T/ipykernel_60736/3169537871.py:44: Pandas4Warning: Using a fill_value that cannot be held in the existing dtype is deprecated and will raise in a future version.
  wide = series.uns

date lastbuy window 36000 done; part.shape= (14963, 1498)
整行缺失客户数(信用板块特征将置为-999): 219 / 14963
creditos_features.shape = (14963, 31461)


,apply_id,cust_no,create_time,cr_upd_win_30d_total_cnt,cr_upd_all_30d_overdue_rate,cr_upd_all_30d_avg_limit,cr_upd_all_30d_cnt,cr_upd_all_30d_limit_pos_cnt,cr_upd_all_30d_overdue_cnt,cr_upd_all_30d_overdue_ratio,...,cr_lastbuy_no_Service_36000d_numPag_eq1_cnt,cr_lastbuy_no_Service_36000d_numPag_eq1_ratio,cr_lastbuy_no_Service_36000d_numPag_1to5_cnt,cr_lastbuy_no_Service_36000d_numPag_1to5_ratio,cr_lastbuy_no_Service_36000d_peor_ge10_cnt,cr_lastbuy_no_Service_36000d_peor_ge10_ratio,cr_lastbuy_no_Service_36000d_peor_ge1_cnt,cr_lastbuy_no_Service_36000d_peor_ge1_ratio,cr_lastbuy_no_Service_36000d_peor_ge2_cnt,cr_lastbuy_no_Service_36000d_peor_ge2_ratio
1355047166686003201,1355047166686003201,800001818243,2026-03-02 10:44:36.872,15,6.268529,226.666672,15,2,4,0.266667,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0
1355151139254296577,1355151139254296577,800001804693,2026-03-02 11:35:04.922,5,1.605499,1942.199951,5,1,5,1.000000,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0
1355334860775358465,1355334860775358465,800001818116,2026-03-02 13:04:13.012,5,1.059171,4100.000000,5,3,3,0.600000,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0


In [7]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

# =========================
# 信用板块（creditos）衍生：输出特征字典 + 导出结果CSV
# - 特征名：cdc_{原特征名}_v3
# - 字典列：feature_name / cn_desc(全中文) / logic_cn(全中文) / missing_rate / nunique / iv_fpd7 / iv_cycle_pass / iv_single_pass
# - IV：保留4位小数，按 iv_cycle_pass 降序排序（缺失放最后）
# =========================

out_dir = Path('outputs')
out_dir.mkdir(parents=True, exist_ok=True)

# 1) 导出衍生结果（文件非常大：约 12546×35029；这里按你的要求导出为 CSV）
result_path = out_dir / 'creditos_features_cdc_v3_up.csv'
creditos_features.to_csv(result_path, index=False, encoding='utf-8-sig')
print('已导出信用板块衍生结果CSV:', result_path, 'shape=', creditos_features.shape)

# 2) 特征字典基础三列
FIELD_CN = {
    'cutoff_dt': '截止日期',
    'request_time': '截止日期（请求时间）',
    'apply_time': '截止日期（申请时间）',
    'fechaActualizacion': '更新日期',
    'fechaReporte': '报告日期',
    'fechaAperturaCuenta': '账户开立日期',
    'fechaUltimaCompra': '最后消费日期',
    'saldoVencido': '逾期余额',
    'limiteCredito': '信用额度',
    'saldoActual': '当前剩余待还金额',
    'montoPagar': '当期应还金额',
    'valorActivoValuacion': '资产估值金额',
    'numeroPagos': '付款次数',
    'peorAtraso': '最严重逾期（最大拖欠程度）',
    'tipoResponsabilidad': '责任类型',
    'frecuenciaPagos': '还款频率',
    'tipoCuenta': '账户类型',
    'tipoCredito': '信用/贷款类型',
    'nombreOtorgante': '机构类别',
}

TAG_CN = {
    'upd': '更新日期口径',
    'open': '开立日期口径',
    'lastbuy': '最后消费日期口径',
}

TAG_EVENT_CN = {
    'upd': f"fechaActualizacion({FIELD_CN['fechaActualizacion']}) 优先，缺失用 fechaReporte({FIELD_CN['fechaReporte']}) 兜底",
    'open': f"fechaAperturaCuenta({FIELD_CN['fechaAperturaCuenta']})",
    'lastbuy': f"fechaUltimaCompra({FIELD_CN['fechaUltimaCompra']})",
}

GROUP_CN = {
    'resp': f"tipoResponsabilidad({FIELD_CN['tipoResponsabilidad']})归类",
    'freq': f"frecuenciaPagos({FIELD_CN['frecuenciaPagos']})归类",
    'ta': f"tipoCuenta({FIELD_CN['tipoCuenta']})归类",
    'tc': f"tipoCredito({FIELD_CN['tipoCredito']})归类",
    'no': f"nombreOtorgante({FIELD_CN['nombreOtorgante']})归类",
    'all': '不分类型（全量）',
    'win': '窗口总体',
}

METRIC_CN = {
    'overdue_rate': '金额逾期率',
    'avg_limit': '平均信用额度',
    'cnt': '数量',
    'limit_pos_cnt': '有效额度数量',
    'overdue_cnt': '逾期数量',
    'overdue_ratio': '逾期占比',
    'ratio': '占比',
    'sum_saldoActual': '总剩余待还金额',
    'sum_limiteCredito': '总信用额度',
    'sum_saldoVencido': '总逾期剩余待还金额',
    'max_saldoActual': '最大剩余待还金额',
    'max_limiteCredito': '最大信用额度',
    'sum_montoPagar': '最近应还余额不含坏账（当期应还金额求和）',
    'paid_off_cnt': '还清借款数量',
    'paid_off_ratio': '还清借款占比',
    'borrow_total_cnt': '借款总数（全量总笔数）',
    'unpaid_cnt': '未还清借款数量',
    'unpaid_ratio': '未还清借款占比',
    'asset_gt10k_cnt': '资产价值大于10K数量',
    'asset_gt10k_ratio': '资产价值大于10K占比',
    'asset_gt1k_cnt': '资产价值大于1K数量',
    'asset_gt1k_ratio': '资产价值大于1K占比',
    'sum_numeroPagos': '总还款次数',
    'numPag_ge5_cnt': '总还款次数大于等于5数量',
    'numPag_ge5_ratio': '总还款次数大于等于5占比',
    'numPag_eq1_cnt': '总还款次数等于1数量',
    'numPag_eq1_ratio': '总还款次数等于1占比',
    'numPag_1to5_cnt': '总还款次数在1-5之间数量',
    'numPag_1to5_ratio': '总还款次数在1-5之间占比',
    'peor_ge10_cnt': '最大逾期大于等于10数量',
    'peor_ge10_ratio': '最大逾期大于等于10占比',
    'peor_ge1_cnt': '最大逾期大于等于1数量',
    'peor_ge1_ratio': '最大逾期大于等于1占比',
    'peor_ge2_cnt': '最大逾期大于等于2数量',
    'peor_ge2_ratio': '最大逾期大于等于2占比',
    'total_cnt': '窗口总笔数',
}


def window_cn(w: int) -> str:
    return '全量历史（近似）' if int(w) >= 36000 else f'近{int(w)}天'


def base_logic(tag: str, w: int) -> str:
    cutoff_cn = FIELD_CN.get(cutoff_col, '截止日期')
    return (
        f"以{cutoff_col}({cutoff_cn})为截止日期，选择事件日期={TAG_EVENT_CN[tag]}，计算diff_days=截止日期-事件日期(天)，"
        f"筛选diff_days在[0,{w}]内的信用记录（{window_cn(w)}）后统计；"
        f"金额字段 {FIELD_CN['saldoVencido']}/{FIELD_CN['limiteCredito']}/{FIELD_CN['saldoActual']}/{FIELD_CN['montoPagar']}/{FIELD_CN['valorActivoValuacion']}/{FIELD_CN['numeroPagos']}/{FIELD_CN['peorAtraso']} 计算时空值按0参与（不落地填充）。"
    )


def mk_row(col: str):
    if col == 'apply_id':
        return None

    feat_name = f'cdc_{col}_v3'

    # cr_{tag}_win_{w}d_total_cnt
    m = re.match(r'^cr_(upd|open|lastbuy)_win_(\d+)d_total_cnt$', col)
    if m:
        tag, w = m.group(1), int(m.group(2))
        cn_desc = f"{TAG_CN[tag]}{window_cn(w)}信用板块：{METRIC_CN['total_cnt']}"
        logic = base_logic(tag, w) + ' 统计窗口内信用记录总笔数。'
        return feat_name, cn_desc, logic

    # cr_{tag}_all_{w}d_{metric}
    m = re.match(r'^cr_(upd|open|lastbuy)_all_(\d+)d_(.+)$', col)
    if m:
        tag, w, metric = m.group(1), int(m.group(2)), m.group(3)
        metric_cn = METRIC_CN.get(metric, metric)
        cn_desc = f"{TAG_CN[tag]}{window_cn(w)}信用板块：{metric_cn}（不分类型）"

        if metric == 'ratio':
            logic = base_logic(tag, w) + ' 占比=窗口内总笔数/全量(36000窗口)总笔数。'
        elif metric.endswith('_ratio'):
            logic = base_logic(tag, w) + f" {metric_cn}=对应数量/窗口内总笔数。"
        elif metric == 'overdue_rate':
            logic = base_logic(tag, w) + f" {metric_cn}=窗口内saldoVencido({FIELD_CN['saldoVencido']})求和/limiteCredito({FIELD_CN['limiteCredito']})求和。"
        elif metric == 'avg_limit':
            logic = base_logic(tag, w) + f" {metric_cn}=窗口内limiteCredito({FIELD_CN['limiteCredito']})求和/窗口内总笔数。"
        else:
            logic = base_logic(tag, w) + f" 计算{metric_cn}。"
        return feat_name, cn_desc, logic

    # cr_{tag}_{grp}_{Class}_{w}d_{metric}
    m = re.match(r'^cr_(upd|open|lastbuy)_(resp|freq|ta|tc|no)_([^_]+)_(\d+)d_(.+)$', col)
    if m:
        tag, grp, cls, w, metric = m.group(1), m.group(2), m.group(3), int(m.group(4)), m.group(5)
        metric_cn = METRIC_CN.get(metric, metric)
        cn_desc = f"{TAG_CN[tag]}{window_cn(w)}信用板块：{GROUP_CN[grp]}[{cls}]{metric_cn}"

        if metric == 'ratio':
            logic = base_logic(tag, w) + f" 在{GROUP_CN[grp]}中取{cls}子集，{metric_cn}=窗口内该类总笔数/全量(36000窗口)总笔数。"
        elif metric.endswith('_ratio'):
            logic = base_logic(tag, w) + f" 在{GROUP_CN[grp]}中取{cls}子集，{metric_cn}=对应数量/窗口内该类总笔数。"
        elif metric == 'overdue_rate':
            logic = base_logic(tag, w) + (
                f" 在{GROUP_CN[grp]}中取{cls}子集，{metric_cn}=窗口内saldoVencido({FIELD_CN['saldoVencido']})求和/"
                f"limiteCredito({FIELD_CN['limiteCredito']})求和。"
            )
        elif metric == 'avg_limit':
            logic = base_logic(tag, w) + f" 在{GROUP_CN[grp]}中取{cls}子集，{metric_cn}=窗口内limiteCredito({FIELD_CN['limiteCredito']})求和/窗口内该类总笔数。"
        else:
            logic = base_logic(tag, w) + f" 在{GROUP_CN[grp]}中取{cls}子集，计算{metric_cn}。"

        return feat_name, cn_desc, logic

    # 兜底
    return feat_name, '信用板块：特征（未识别命名模式）', '该特征未匹配到预设命名模式，请人工检查其命名与生成逻辑。'


rows = []
for c in creditos_features.columns:
    r = mk_row(c)
    if r is not None:
        rows.append(r)

feature_dict = pd.DataFrame(rows, columns=['feature_name', 'cn_desc', 'logic_cn'])

# 3) 追加 missing_rate / nunique / IV
label_df = df_raw[['apply_id', 'fpd7', 'approve_state']].copy()
label_df['apply_id'] = label_df['apply_id'].astype(str)

work = creditos_features.merge(label_df, on='apply_id', how='left')
y = pd.to_numeric(work['fpd7'], errors='coerce')
mask_y_valid = y.isin([0, 1])
work = work.loc[mask_y_valid].copy()
y = y.loc[mask_y_valid].astype(np.int8).to_numpy()

mask_cycle = (work['approve_state'] == 'CYCLE_PASS').to_numpy()
mask_single = (work['approve_state'] == 'SINGLE_PASS').to_numpy()

feat_cols = [c for c in creditos_features.columns if c != 'apply_id']


def calc_iv_fast(x: np.ndarray, y01: np.ndarray, bins: int = 10) -> float:
    # x: float64, y01: 0/1 int
    if x.size == 0:
        return np.nan
    bad_total = float(y01.sum())
    good_total = float((1 - y01).sum())
    if bad_total == 0 or good_total == 0:
        return np.nan

    miss = np.isnan(x) | (x == -999)
    x_non = x[~miss]

    # 只有缺失 or 常数
    if x_non.size == 0 or np.nanmax(x_non) == np.nanmin(x_non):
        bin_id = np.zeros_like(y01, dtype=np.int16)
        bin_id[~miss] = 1
        n_bins = 2
    else:
        qs = np.linspace(0, 1, bins + 1)
        edges = np.nanquantile(x_non, qs)
        edges = np.unique(edges)
        if edges.size <= 2:
            bin_id = np.zeros_like(y01, dtype=np.int16)
            bin_id[~miss] = 1
            n_bins = 2
        else:
            inner = edges[1:-1]
            b = np.digitize(x_non, inner, right=True)
            bin_id = np.zeros_like(y01, dtype=np.int16)
            bin_id[~miss] = (b + 1).astype(np.int16)
            n_bins = int(bin_id.max() + 1)

    bad = np.bincount(bin_id, weights=y01, minlength=n_bins).astype('float64')
    good = np.bincount(bin_id, weights=(1 - y01), minlength=n_bins).astype('float64')

    eps = 1e-6
    dist_bad = np.clip(bad / bad_total, eps, None)
    dist_good = np.clip(good / good_total, eps, None)
    woe = np.log(dist_bad / dist_good)
    iv = ((dist_bad - dist_good) * woe).sum()
    return float(iv)


missing_rate_list = []
nunique_list = []
iv_all_list = []
iv_cycle_list = []
iv_single_list = []

X = work[feat_cols]  # pandas DataFrame

for i, c in enumerate(feat_cols, 1):
    s = X[c]
    miss = s.isna() | (s == -999)
    missing_rate_list.append(float(miss.mean()))
    nunique_list.append(int(s.mask(s == -999).nunique(dropna=True)))

    x = pd.to_numeric(s, errors='coerce').to_numpy(dtype='float64')

    iv_all_list.append(calc_iv_fast(x, y))
    iv_cycle_list.append(calc_iv_fast(x[mask_cycle], y[mask_cycle]))
    iv_single_list.append(calc_iv_fast(x[mask_single], y[mask_single]))

    if i % 500 == 0:
        print(f'IV进度: {i}/{len(feat_cols)}')

feature_dict['missing_rate'] = missing_rate_list
feature_dict['nunique'] = nunique_list
feature_dict['iv_fpd7'] = pd.to_numeric(pd.Series(iv_all_list), errors='coerce').round(4)
feature_dict['iv_cycle_pass'] = pd.to_numeric(pd.Series(iv_cycle_list), errors='coerce').round(4)
feature_dict['iv_single_pass'] = pd.to_numeric(pd.Series(iv_single_list), errors='coerce').round(4)

# 按 iv_cycle_pass 从大到小排序
feature_dict = feature_dict.sort_values(by='iv_cycle_pass', ascending=False, na_position='last', kind='mergesort').reset_index(drop=True)

# 4) 保存字典
feat_dict_path = out_dir / 'creditos_features_dict_cdc_v3_up.csv'
feature_dict.to_csv(feat_dict_path, index=False, encoding='utf-8-sig')
print('已导出信用板块特征字典CSV:', feat_dict_path, 'rows=', len(feature_dict))

feature_dict.head(20)

已导出信用板块衍生结果CSV: outputs/creditos_features_cdc_v3_up.csv shape= (14963, 31461)
IV进度: 500/31460
IV进度: 1000/31460
IV进度: 1500/31460
IV进度: 2000/31460
IV进度: 2500/31460
IV进度: 3000/31460
IV进度: 3500/31460
IV进度: 4000/31460
IV进度: 4500/31460
IV进度: 5000/31460
IV进度: 5500/31460
IV进度: 6000/31460
IV进度: 6500/31460
IV进度: 7000/31460
IV进度: 7500/31460
IV进度: 8000/31460
IV进度: 8500/31460
IV进度: 9000/31460
IV进度: 9500/31460
IV进度: 10000/31460
IV进度: 10500/31460
IV进度: 11000/31460
IV进度: 11500/31460
IV进度: 12000/31460
IV进度: 12500/31460
IV进度: 13000/31460
IV进度: 13500/31460
IV进度: 14000/31460
IV进度: 14500/31460
IV进度: 15000/31460
IV进度: 15500/31460
IV进度: 16000/31460
IV进度: 16500/31460
IV进度: 17000/31460
IV进度: 17500/31460
IV进度: 18000/31460
IV进度: 18500/31460
IV进度: 19000/31460
IV进度: 19500/31460
IV进度: 20000/31460
IV进度: 20500/31460
IV进度: 21000/31460
IV进度: 21500/31460
IV进度: 22000/31460
IV进度: 22500/31460
IV进度: 23000/31460
IV进度: 23500/31460
IV进度: 24000/31460
IV进度: 24500/31460
IV进度: 25000/31460
IV进度: 25500/31460
IV进度: 26000/31460
IV进度: 

,feature_name,cn_desc,logic_cn,missing_rate,nunique,iv_fpd7,iv_cycle_pass,iv_single_pass
0,cdc_cust_no_v3,信用板块：特征（未识别命名模式）,该特征未匹配到预设命名模式，请人工检查其命名与生成逻辑。,NaN,0,NaN,NaN,NaN
1,cdc_create_time_v3,信用板块：特征（未识别命名模式）,该特征未匹配到预设命名模式，请人工检查其命名与生成逻辑。,NaN,0,NaN,NaN,NaN
2,cdc_cr_upd_win_30d_total_cnt_v3,更新日期口径近30天信用板块：窗口总笔数,以apply_time(截止日期（申请时间）)为截止日期，选择事件日期=fechaActua...,NaN,0,NaN,NaN,NaN
3,cdc_cr_upd_all_30d_overdue_rate_v3,更新日期口径近30天信用板块：金额逾期率（不分类型）,以apply_time(截止日期（申请时间）)为截止日期，选择事件日期=fechaActua...,NaN,0,NaN,NaN,NaN
4,cdc_cr_upd_all_30d_avg_limit_v3,更新日期口径近30天信用板块：平均信用额度（不分类型）,以apply_time(截止日期（申请时间）)为截止日期，选择事件日期=fechaActua...,NaN,0,NaN,NaN,NaN
5,cdc_cr_upd_all_30d_cnt_v3,更新日期口径近30天信用板块：数量（不分类型）,以apply_time(截止日期（申请时间）)为截止日期，选择事件日期=fechaActua...,NaN,0,NaN,NaN,NaN
6,cdc_cr_upd_all_30d_limit_pos_cnt_v3,更新日期口径近30天信用板块：有效额度数量（不分类型）,以apply_time(截止日期（申请时间）)为截止日期，选择事件日期=fechaActua...,NaN,0,NaN,NaN,NaN
7,cdc_cr_upd_all_30d_overdue_cnt_v3,更新日期口径近30天信用板块：逾期数量（不分类型）,以apply_time(截止日期（申请时间）)为截止日期，选择事件日期=fechaActua...,NaN,0,NaN,NaN,NaN
8,cdc_cr_upd_all_30d_overdue_ratio_v3,更新日期口径近30天信用板块：逾期占比（不分类型）,以apply_time(截止日期（申请时间）)为截止日期，选择事件日期=fechaActua...,NaN,0,NaN,NaN,NaN
9,cdc_cr_upd_all_30d_ratio_v3,更新日期口径近30天信用板块：占比（不分类型）,以apply_time(截止日期（申请时间）)为截止日期，选择事件日期=fechaActua...,NaN,0,NaN,NaN,NaN
